# Laboratorio 4 — Segmentación de imágenes con técnicas de agrupación
## Caso: DermAlpes — Delimitación automática de lesiones cutáneas en imágenes dermatoscópicas

**Curso:** ISIS-2611 Aprendizaje de Máquina — Universidad de los Andes

**Integrantes:**
- Julian Pinto — 202321207
- Daniel Vergara — 202320392

---

### Contexto del problema y objetivos
**DermAlpes** es una empresa de telemedicina que conecta centros de salud de municipios pequeños con dermatólogos ubicados en las ciudades principales. En los centros de salud, el personal toma fotografías de lesiones de piel con un dermatoscopio y las envía para revisión remota. Para agilizar este proceso, DermAlpes busca incorporar una herramienta de apoyo que **delimite automáticamente la zona de la lesión** en cada imagen, permitiendo al especialista medir su tamaño y forma y concentrar su atención en la región relevante.

Dado que la empresa no cuenta con imágenes etiquetadas de sus propios pacientes, se evalúa un enfoque de **aprendizaje no supervisado** basado en **K-medias**, agrupando los píxeles de cada imagen según su **color** $[R, G, B]$ y su **posición espacial** $[x, y]$. La herramienta **no realiza diagnósticos**; su único propósito es delimitar la lesión como apoyo visual.

### Fuente del conjunto de datos
Se utiliza una selección de 20 imágenes dermatoscópicas del desafío **ISIC 2016** (*Task 1: Lesion Segmentation*), acompañadas de sus respectivas máscaras de referencia elaboradas por dermatólogos especialistas:
> **Cita:** Gutman, D., Codella, N. C. F., Celebi, E., Helba, B., Marchetti, M., Mishra, N., & Halpern, A. (2016). *Skin Lesion Analysis toward Melanoma Detection: A Challenge at the International Symposium on Biomedical Imaging (ISBI) 2016, hosted by the International Skin Imaging Collaboration (ISIC)*. arXiv:1605.01397. Disponible en: https://challenge.isic-archive.com/landing/2016/37/

### Condición estricta del protocolo experimental
- **Conjunto de ajuste (15 imágenes):** se utiliza para explorar las combinaciones de pesos $(\beta, \lambda)$, seleccionar el número de grupos $K$ mediante el coeficiente de silueta, diseñar y comparar los criterios de construcción de la máscara binaria y elegir la configuración final mediante el coeficiente de Dice promedio.
- **Conjunto de prueba (5 imágenes):** se reserva **exclusivamente** para la evaluación final (Actividad 5). Sus imágenes y máscaras **no se consultan ni se utilizan** para elegir $\beta$, $\lambda$, $K$ ni para ajustar el criterio de construcción de la máscara.

---

### Contenido del Notebook
0. [Importación de librerías y configuración global](#sec0)
1. [Carga de datos, preparación de máscaras y exploración visual](#sec1)
2. [Actividad 1: Pipeline de preparación y representación `[β·R, β·G, β·B, λ·x, λ·y]`](#sec2)
3. [Actividad 2: Agrupación con K-medias y selección de K mediante silueta](#sec3)
4. [Actividad 3: Construcción automática de la máscara de la lesión](#sec4)
5. [Actividad de verificación obligatoria con IAG](#sec5)
6. [Actividad 4: Ajuste del método sobre las 15 imágenes de ajuste](#sec6)
7. [Actividad 5: Evaluación final y análisis sobre las 5 imágenes de prueba](#sec7)
8. [Análisis de resultados (Preguntas 1 a 5)](#sec8)
9. [Uso de herramientas de IA generativa](#sec9)
10. [Bonos opcionales (Bono A: PCA/t-SNE y Bono B: Mean Shift)](#sec10)

## <a name="sec0"></a> 0. Importación de librerías y configuración global

Se importan las librerías requeridas para el procesamiento de imágenes, construcción del pipeline en `scikit-learn`, algoritmos de agrupación, métricas de evaluación interna/externa y visualización, siguiendo la estructura de las prácticas **P11 (K-Means)**, **P12 (DBSCAN y Aglomerativo)** y **P13 (Segmentación de imágenes con K-Means)**.

In [ ]:
import os
import time
import warnings
from glob import glob
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from PIL import Image
from IPython.display import Markdown, display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans, MeanShift, estimate_bandwidth
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    f1_score,
    v_measure_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

# Opcional: OpenCV o Plotly si están instalados en el entorno
try:
    import cv2
    HAS_CV2 = True
except ImportError:
    HAS_CV2 = False

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')

print(f"Versión de NumPy:        {version('numpy')}")
print(f"Versión de Pandas:       {version('pandas')}")
print(f"Versión de Matplotlib:   {version('matplotlib')}")
print(f"Versión de Scikit-Learn: {version('scikit-learn')}")
print(f"Versión de Pillow (PIL): {version('pillow')}")
print(f"OpenCV disponible:       {HAS_CV2} | Plotly disponible: {HAS_PLOTLY}")

In [ ]:
# ==============================================================================
# CONSTANTES DEL PROTOCOLO Y PARTICIÓN OFICIAL DEL CONJUNTO DE DATOS
# ==============================================================================
RANDOM_STATE = 42

# Tamaño de la muestra reproducible de píxeles para calcular el coeficiente de silueta
# (Actividad 2: "calculado sobre una muestra reproducible de píxeles. Documenten el tamaño de la muestra y la semilla utilizada")
SAMPLE_SIZE_SILUETA = 3000

# Resolución de trabajo para las imágenes (ancho, alto) o None para mantener resolución original.
# Nota: Redimensionar a (256, 256) o (200, 200) conserva la morfología de la lesión y permite
# ejecutar las 15 imágenes x 10 combinaciones x 7 valores de K en pocos minutos.
# Las máscaras se redimensionan SIEMPRE con vecino más cercano (NEAREST) para conservar {0, 1}.
TARGET_SIZE = (256, 256)  # Cambiar a None si se desea usar el tamaño original completo

# Rango de K a explorar según el enunciado (K entre 2 y 8)
K_RANGE = list(range(2, 9))

# 15 Imágenes de ajuste (definidas en el enunciado)
IMAGENES_AJUSTE = [
    'ISIC_0000085', 'ISIC_0000114', 'ISIC_0000150',
    'ISIC_0000086', 'ISIC_0000116', 'ISIC_0000153',
    'ISIC_0000089', 'ISIC_0000122', 'ISIC_0000156',
    'ISIC_0000094', 'ISIC_0000133', 'ISIC_0000157',
    'ISIC_0000108', 'ISIC_0000137', 'ISIC_0000112',
]

# 5 Imágenes de prueba (reservadas exclusivamente para la Actividad 5)
IMAGENES_PRUEBA = [
    'ISIC_0000087',
    'ISIC_0000105',
    'ISIC_0000110',
    'ISIC_0000123',
    'ISIC_0000128',
]

assert len(IMAGENES_AJUSTE) == 15, "Deben ser exactamente 15 imágenes de ajuste."
assert len(IMAGENES_PRUEBA) == 5, "Deben ser exactamente 5 imágenes de prueba."
assert set(IMAGENES_AJUSTE).isdisjoint(set(IMAGENES_PRUEBA)), "No debe haber solapamiento entre ajuste y prueba."

# Directorio raíz donde se ubican las imágenes y máscaras
DATA_DIR = Path('data')
print(f"Configuración lista: {len(IMAGENES_AJUSTE)} imágenes de ajuste | {len(IMAGENES_PRUEBA)} imágenes de prueba.")
print(f"Semilla global (RANDOM_STATE): {RANDOM_STATE} | Muestra de silueta: {SAMPLE_SIZE_SILUETA} píxeles.")

## <a name="sec1"></a> 1. Carga de datos, preparación de máscaras y exploración visual

Siguiendo las instrucciones de la sección **Conjunto de datos**:
1. **Lectura en RGB:** Cada imagen dermatoscópica se carga en formato RGB con valores en $[0, 255]$.
2. **Preparación de las máscaras de referencia:** Las máscaras originales contienen valores de `0` (piel o fondo) y `255` (lesión), los cuales se convierten a valores binarios `0` y `1`, respectivamente.
3. **Consistencia dimensional e interpolación:** Si se reduce el tamaño de las imágenes (`TARGET_SIZE`), las máscaras de referencia se redimensionan exactamente a las mismas dimensiones utilizando **interpolación por vecino más cercano** (`Image.Resampling.NEAREST`), garantizando que no aparezcan valores intermedios y se conserven estrictamente sus valores binarios $\{0, 1\}$.

In [ ]:
def encontrar_archivos_isic(isic_id, data_dir=DATA_DIR):
    '''
    Busca automáticamente el archivo de imagen (.jpg/.png) y su máscara de referencia
    (_Segmentation.png / mask) dentro de `data_dir` y sus subdirectorios.
    '''
    data_dir = Path(data_dir)
    if not data_dir.exists():
        raise FileNotFoundError(
            f"No se encontró el directorio '{data_dir}'. "
            f"Por favor coloca las 20 imágenes y sus máscaras en la carpeta '{data_dir.resolve()}'."
        )

    candidatos = list(data_dir.rglob(f"*{isic_id}*"))
    if not candidatos:
        raise FileNotFoundError(f"No se encontraron archivos para '{isic_id}' dentro de '{data_dir}'.")

    ruta_img, ruta_mask = None, None
    for p in sorted(candidatos):
        if not p.is_file():
            continue
        nombre_lower = p.name.lower()
        es_mascara = (
            'seg' in nombre_lower
            or 'mask' in nombre_lower
            or 'mascara' in nombre_lower
            or 'mask' in str(p.parent).lower()
            or 'seg' in str(p.parent).lower()
        )
        if es_mascara:
            ruta_mask = p
        else:
            ruta_img = p

    if ruta_img is None or ruta_mask is None:
        raise FileNotFoundError(
            f"Para '{isic_id}' se encontró img={ruta_img}, mask={ruta_mask}. "
            f"Verifica que tanto la imagen como su máscara estén en '{data_dir}'."
        )
    return ruta_img, ruta_mask


def cargar_imagen_y_mascara(isic_id, data_dir=DATA_DIR, target_size=TARGET_SIZE):
    '''
    Carga una imagen dermatoscópica en RGB y su máscara de referencia binaria {0, 1}.
    Si `target_size` = (W, H) está definido, redimensiona la imagen con interpolación bilineal
    y la máscara con interpolación por VECINO MÁS CERCANO (NEAREST) para preservar {0, 1}.
    '''
    ruta_img, ruta_mask = encontrar_archivos_isic(isic_id, data_dir=data_dir)

    with Image.open(ruta_img) as pil_img:
        pil_img = pil_img.convert('RGB')
        orig_w, orig_h = pil_img.size
        if target_size is not None:
            pil_img = pil_img.resize(target_size, resample=Image.Resampling.BILINEAR)
        img_rgb = np.asarray(pil_img, dtype=np.uint8)

    with Image.open(ruta_mask) as pil_mask:
        pil_mask = pil_mask.convert('L')
        if target_size is not None:
            # Interpolación por vecino más cercano como exige el enunciado
            pil_mask = pil_mask.resize(target_size, resample=Image.Resampling.NEAREST)
        mask_raw = np.asarray(pil_mask, dtype=np.uint8)

    # Conversión de {0, 255} a {0, 1} (0 = piel/fondo, 1 = lesión)
    mask_bin = (mask_raw >= 128).astype(np.uint8)

    # Verificaciones de integridad
    assert img_rgb.shape[:2] == mask_bin.shape, (
        f"Dimensiones incompatibles en {isic_id}: img {img_rgb.shape[:2]} vs mask {mask_bin.shape}"
    )
    assert set(np.unique(mask_bin)).issubset({0, 1}), (
        f"La máscara de {isic_id} no es estrictamente binaria: {np.unique(mask_bin)}"
    )

    return {
        'id': isic_id,
        'img': img_rgb,
        'mask': mask_bin,
        'original_shape': (orig_h, orig_w),
        'shape': img_rgb.shape[:2],
        'ruta_img': str(ruta_img),
        'ruta_mask': str(ruta_mask),
    }


def cargar_conjunto_isic(lista_ids, data_dir=DATA_DIR, target_size=TARGET_SIZE):
    '''Carga un diccionario {isic_id: dict_datos} para la lista de identificadores dada.'''
    conjunto = {}
    for isic_id in lista_ids:
        conjunto[isic_id] = cargar_imagen_y_mascara(isic_id, data_dir=data_dir, target_size=target_size)
    return conjunto

In [ ]:
# Carga de los conjuntos de ajuste (15 imágenes) y prueba (5 imágenes)
# Nota: El conjunto de prueba solo se carga para verificar la existencia de los archivos;
# NO se explora ni se consulta hasta la Actividad 5.
try:
    datos_ajuste = cargar_conjunto_isic(IMAGENES_AJUSTE, data_dir=DATA_DIR, target_size=TARGET_SIZE)
    datos_prueba = cargar_conjunto_isic(IMAGENES_PRUEBA, data_dir=DATA_DIR, target_size=TARGET_SIZE)
    DATOS_CARGADOS = True
    print(f"Conjunto de ajuste cargado exitosamente: {len(datos_ajuste)} imágenes.")
    print(f"Conjunto de prueba verificado (reservado para Act. 5): {len(datos_prueba)} imágenes.")
except FileNotFoundError as e:
    DATOS_CARGADOS = False
    print(f"[AVISO] {e}")

### 1.1 Exploración descriptiva y visual del conjunto de ajuste (15 imágenes)

A continuación, se inspeccionan exclusivamente las **15 imágenes de ajuste** para conocer sus dimensiones originales, la proporción del área ocupada por la lesión, el contraste cromático entre lesión y fondo, y posibles artefactos visuales (vello, viñeteo circular del dermatoscopio, burbujas de gel, marcas o iluminación no uniforme).

In [ ]:
def resumen_exploratorio_ajuste(datos_dict):
    '''Construye una tabla descriptiva de las imágenes del conjunto de ajuste.'''
    filas = []
    for isic_id, info in datos_dict.items():
        img = info['img']
        mask = info['mask']
        h_orig, w_orig = info['original_shape']
        h, w = info['shape']
        pix_lesion = img[mask == 1]
        pix_fondo = img[mask == 0]
        filas.append({
            'Imagen': isic_id,
            'Alto orig.': h_orig,
            'Ancho orig.': w_orig,
            'Resolución trabajo': f"{h}x{w}",
            '% Píxeles lesión': round(mask.mean() * 100, 2),
            'RGB medio (lesión)': np.round(pix_lesion.mean(axis=0), 1).tolist() if len(pix_lesion) else None,
            'RGB medio (fondo)': np.round(pix_fondo.mean(axis=0), 1).tolist() if len(pix_fondo) else None,
            'Luminancia lesión': round(pix_lesion.mean(), 1) if len(pix_lesion) else np.nan,
            'Luminancia fondo': round(pix_fondo.mean(), 1) if len(pix_fondo) else np.nan,
        })
    return pd.DataFrame(filas)


def mostrar_conjunto_con_mascaras(datos_dict, titulo="Imágenes de ajuste y contorno de la máscara de referencia", cols=5):
    '''
    Adaptado de `show_imgs` (P13): muestra cada imagen junto con el contorno de su máscara
    de referencia superpuesto en color verde y el porcentaje de área de lesión.
    '''
    items = list(datos_dict.values())
    n = len(items)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.2 * cols, 3.2 * rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, item in zip(axes, items):
        ax.imshow(item['img'])
        ax.contour(item['mask'], levels=[0.5], colors=['#00FF66'], linewidths=1.5)
        pct = item['mask'].mean() * 100
        ax.set_title(f"{item['id']}\n({pct:.1f}% lesión)", fontsize=9)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle(titulo, fontsize=13, fontweight='bold')
    plt.show()


def plot_data_rgb_3d(img, mask=None, pct=0.02, random_state=RANDOM_STATE, titulo="Distribución de píxeles en el espacio RGB"):
    '''
    Adaptado de `plot_data` (P13_Segmentación de imágenes):
    Visualiza una muestra reproducible de píxeles en el espacio 3D (R, G, B).
    '''
    rng = np.random.default_rng(random_state)
    X = img.reshape(-1, 3)
    n_sample = max(500, min(len(X), round(len(X) * pct)))
    idx = rng.choice(len(X), size=n_sample, replace=False)
    X_s = X[idx]

    fig = plt.figure(figsize=(11, 4.8))
    ax1 = fig.add_subplot(1, 2 if mask is not None else 1, 1, projection='3d')
    ax1.scatter(X_s[:, 0], X_s[:, 1], X_s[:, 2], c=X_s / 255.0, s=4, alpha=0.7)
    ax1.set(xlabel='R', ylabel='G', zlabel='B', title='Color real de cada píxel')

    if mask is not None:
        m_s = mask.reshape(-1)[idx]
        ax2 = fig.add_subplot(1, 2, 2, projection='3d')
        colores_ref = np.where(m_s == 1, '#E03131', '#4C6EF5')
        ax2.scatter(X_s[:, 0], X_s[:, 1], X_s[:, 2], c=colores_ref, s=4, alpha=0.6)
        ax2.set(xlabel='R', ylabel='G', zlabel='B', title='Máscara de referencia (Rojo=Lesión, Azul=Fondo)')

    plt.suptitle(titulo, fontsize=12)
    plt.tight_layout()
    plt.show()


if DATOS_CARGADOS:
    tabla_exploratoria = resumen_exploratorio_ajuste(datos_ajuste)
    display(tabla_exploratoria)
    mostrar_conjunto_con_mascaras(datos_ajuste, "Conjunto de Ajuste (15 imágenes) — Contorno verde: máscara de referencia")
    # Ejemplo de distribución 3D en RGB sobre la primera imagen de ajuste (como en P13)
    primera_id = IMAGENES_AJUSTE[0]
    plot_data_rgb_3d(
        datos_ajuste[primera_id]['img'],
        datos_ajuste[primera_id]['mask'],
        titulo=f"Espacio RGB de {primera_id} (muestra reproducible)"
    )

## <a name="sec2"></a> 2. Actividad 1 — Pipeline de preparación y representación (10%)

> **Enunciado:** *Construyan un pipeline de Scikit-learn que represente cada imagen mediante una matriz con un píxel por fila y las características $[\beta \cdot R, \beta \cdot G, \beta \cdot B, \lambda \cdot x, \lambda \cdot y]$, con los valores de color y las coordenadas normalizados al intervalo $[0, 1]$. Los parámetros $\beta$ y $\lambda$ controlan el peso del color y de la posición, respectivamente, y deben poder modificarse sin cambiar la implementación. Cuando $\beta = 0$ y $\lambda \neq 0$, se utiliza únicamente la posición; cuando $\lambda = 0$ y $\beta \neq 0$, únicamente el color.*

### Diseño del Pipeline en Scikit-Learn
Siguiendo el diseño modular de `Pipeline` visto en **P11_KMeans** y la extracción espacial con `np.meshgrid` de **P13_Segmentación de imágenes**, implementamos dos transformadores personalizados que heredan de `BaseEstimator` y `TransformerMixin`:

1. **`ExtractorPixelesImagen`**:
   - Recibe una imagen de dimensiones $(H, W, 3)$ y genera una grilla de coordenadas espaciales $(x, y)$ mediante `np.meshgrid`, donde $x \in \{0, \dots, W-1\}$ (columna horizontal) e $y \in \{0, \dots, H-1\}$ (fila vertical).
   - Normaliza tanto los canales de color $[R, G, B]$ como las coordenadas $[x, y]$ al intervalo $[0, 1]$ (soportando normalización teórica por dominio $R/255, x/(W-1), y/(H-1)$ o empírica por imagen mediante `MinMaxScaler()`).
2. **`PonderadorColorPosicion(beta=1.0, lam=1.0)`**:
   - Multiplica las tres columnas cromáticas por $\beta$ y las dos columnas espaciales por $\lambda$, entregando la matriz $(H \cdot W) \times 5$:
     $$X_{\text{rep}} = [\beta \cdot R,\; \beta \cdot G,\; \beta \cdot B,\; \lambda \cdot x,\; \lambda \cdot y]$$
   - Los hiperparámetros `beta` ($\beta$) y `lam` ($\lambda$) se modifican dinámicamente mediante `pipeline.set_params(ponderador__beta=..., ponderador__lam=...)` sin cambiar la implementación.
   - **Nota geométrica:** En la distancia euclidiana usada por K-medias y por el coeficiente de silueta:
     $$d(p, q) = \sqrt{\beta^2 \left[(R_p - R_q)^2 + (G_p - G_q)^2 + (B_p - B_q)^2\right] + \lambda^2 \left[(x_p - x_q)^2 + (y_p - y_q)^2\right]}$$
     Cuando $\lambda = 0$ y $\beta \neq 0$, las coordenadas valen $0$ para todos los píxeles y la distancia depende **únicamente del color**. Cuando $\beta = 0$ y $\lambda \neq 0$, las componentes de color valen $0$ para todos los píxeles y la distancia depende **únicamente de la posición**.

In [ ]:
class ExtractorPixelesImagen(BaseEstimator, TransformerMixin):
    '''
    Transformador de Scikit-Learn que convierte una imagen RGB de dimensiones (H, W, 3)
    en una matriz 2D de tamaño (H * W, 5) con columnas [R, G, B, x, y] normalizadas al
    intervalo [0, 1].
    '''
    def __init__(self, normalizar_rango_fijo=True):
        self.normalizar_rango_fijo = normalizar_rango_fijo

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def transform(self, X):
        arr = np.asarray(X)
        if arr.ndim != 3 or arr.shape[2] != 3:
            raise ValueError(f"Se esperaba una imagen RGB de forma (H, W, 3), pero se recibió {arr.shape}.")

        H, W, _ = arr.shape
        # Coordenadas: y corresponde a la fila [0, H-1] y x a la columna [0, W-1]
        y_grid, x_grid = np.meshgrid(np.arange(H, dtype=np.float64), np.arange(W, dtype=np.float64), indexing='ij')

        rgb = arr.reshape(-1, 3).astype(np.float64)
        coords = np.column_stack([x_grid.ravel(), y_grid.ravel()])

        if self.normalizar_rango_fijo:
            # Normalización exacta al intervalo [0, 1] según el dominio de color [0, 255] y de coordenadas
            rgb_norm = rgb / 255.0
            x_norm = coords[:, 0] / max(W - 1, 1)
            y_norm = coords[:, 1] / max(H - 1, 1)
            features = np.column_stack([rgb_norm, x_norm, y_norm])
        else:
            # Normalización por mínimo-máximo empírico de cada canal en [0, 1]
            features = MinMaxScaler().fit_transform(np.column_stack([rgb, coords]))

        return features

    def get_feature_names_out(self, input_features=None):
        return np.array(['R', 'G', 'B', 'x', 'y'])


class PonderadorColorPosicion(BaseEstimator, TransformerMixin):
    '''
    Transformador de Scikit-Learn que pondera las características normalizadas [R, G, B, x, y]
    por los pesos beta (color) y lam (posición), produciendo:
        [beta * R, beta * G, beta * B, lam * x, lam * y]
    '''
    def __init__(self, beta=1.0, lam=1.0):
        self.beta = beta
        self.lam = lam

    def fit(self, X, y=None):
        if self.beta < 0 or self.lam < 0:
            raise ValueError("Los pesos beta y lam deben ser mayores o iguales a 0.")
        if self.beta == 0 and self.lam == 0:
            raise ValueError("Al menos uno de los pesos (beta o lam) debe ser distinto de 0.")
        self.is_fitted_ = True
        return self

    def transform(self, X):
        X_arr = np.asarray(X, dtype=np.float64)
        if X_arr.shape[1] != 5:
            raise ValueError(f"Se esperaban 5 columnas [R, G, B, x, y], pero se recibieron {X_arr.shape[1]}.")

        pesos = np.array([self.beta, self.beta, self.beta, self.lam, self.lam], dtype=np.float64)
        return X_arr * pesos

    def get_feature_names_out(self, input_features=None):
        return np.array(['beta_R', 'beta_G', 'beta_B', 'lam_x', 'lam_y'])


def crear_pipeline_representacion(beta=1.0, lam=1.0, normalizar_rango_fijo=True):
    '''
    Construye el Pipeline de Scikit-Learn para representar una imagen como una matriz
    de píxeles con características [beta*R, beta*G, beta*B, lam*x, lam*y].
    '''
    return Pipeline([
        ('extractor', ExtractorPixelesImagen(normalizar_rango_fijo=normalizar_rango_fijo)),
        ('ponderador', PonderadorColorPosicion(beta=beta, lam=lam)),
    ])


def crear_pipeline_completo(beta=1.0, lam=1.0, n_clusters=2, random_state=RANDOM_STATE, normalizar_rango_fijo=True):
    '''
    Construye el Pipeline completo que encadena la preparación/representación de píxeles
    y el modelo de agrupación KMeans, siguiendo el patrón de P11_KMeans.
    '''
    return Pipeline([
        ('extractor', ExtractorPixelesImagen(normalizar_rango_fijo=normalizar_rango_fijo)),
        ('ponderador', PonderadorColorPosicion(beta=beta, lam=lam)),
        ('kmeans', KMeans(n_clusters=n_clusters, n_init=10, max_iter=300, random_state=random_state)),
    ])

In [ ]:
# ==============================================================================
# VERIFICACIÓN DEL PIPELINE DE REPRESENTACIÓN (Actividad 1)
# ==============================================================================
# Creamos una imagen de prueba sintética (o usamos la primera de ajuste si está cargada)
_img_demo = datos_ajuste[IMAGENES_AJUSTE[0]]['img'] if DATOS_CARGADOS else np.random.default_rng(42).integers(0, 256, size=(64, 64, 3), dtype=np.uint8)

pipe_rep = crear_pipeline_representacion(beta=1.0, lam=0.5)

# 1. Verificación en [0, 1] cuando beta=1, lam=1
pipe_rep.set_params(ponderador__beta=1.0, ponderador__lam=1.0)
X_norm = pipe_rep.fit_transform(_img_demo)
assert X_norm.shape == (_img_demo.shape[0] * _img_demo.shape[1], 5)
assert X_norm.min() >= 0.0 and X_norm.max() <= 1.0 + 1e-12

# 2. Verificación de caso extremo: SOLO COLOR (lam = 0, beta != 0)
pipe_rep.set_params(ponderador__beta=1.0, ponderador__lam=0.0)
X_solo_color = pipe_rep.transform(_img_demo)
assert np.allclose(X_solo_color[:, 3:], 0.0), "Con lam=0 las columnas de posición deben ser 0."

# 3. Verificación de caso extremo: SOLO POSICIÓN (beta = 0, lam != 0)
pipe_rep.set_params(ponderador__beta=0.0, ponderador__lam=1.0)
X_solo_pos = pipe_rep.transform(_img_demo)
assert np.allclose(X_solo_pos[:, :3], 0.0), "Con beta=0 las columnas de color deben ser 0."

print("Verificación exitosa del Pipeline de representación:")
print(f" - Dimensiones de imagen de entrada: {_img_demo.shape}")
print(f" - Dimensiones de matriz de salida:  {X_norm.shape} (1 fila por píxel, 5 características)")
display(pd.DataFrame(X_norm[:5], columns=pipe_rep.named_steps['ponderador'].get_feature_names_out()).round(4))

## <a name="sec3"></a> 3. Actividad 2 — Agrupación y selección de K mediante el coeficiente de silueta (10%)

> **Enunciado:** *Para cada imagen de ajuste, exploren diez combinaciones de $\beta$ y $\lambda$ con diferentes pesos relativos, incluyendo una que utilice únicamente el color ($\lambda = 0, \beta \neq 0$) y otra únicamente la posición ($\beta = 0, \lambda \neq 0$). Para cada combinación, apliquen K-medias con $K$ entre 2 y 8 y seleccionen $K$ mediante el coeficiente de silueta, calculado sobre una muestra reproducible de píxeles. Documenten el tamaño de la muestra y la semilla utilizada.*

### Justificación de las 10 combinaciones de $(\beta, \lambda)$
Como señala la observación del enunciado, tanto K-medias como el coeficiente de silueta son invariantes al escalamiento uniforme de todas las coordenadas por una constante positiva, por lo que **lo que determina la geometría de los grupos es la razón $\lambda / \beta$**:
- Fijamos $\beta = 1.0$ para las primeras 9 configuraciones y variamos $\lambda \in \{0.0, 0.10, 0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00\}$, cubriendo desde **únicamente color** ($\lambda/\beta = 0$), pasando por pesos moderados de posición ($\lambda/\beta < 1$), peso equilibrado ($\lambda/\beta = 1$), hasta regímenes dominados por la posición espacial ($\lambda/\beta > 1$).
- La décima configuración fija $(\beta = 0.0, \lambda = 1.0)$ ($\lambda/\beta = \infty$), utilizando **únicamente la posición**.

### Documentación del muestreo para el coeficiente de silueta
- El cálculo exacto de `silhouette_score` tiene complejidad temporal y espacial $\mathcal{O}(N^2)$ en el número de píxeles. En una imagen de $256 \times 256 = 65\,536$ píxeles, calcular la matriz completa de distancias requeriría más de $4.2 \times 10^9$ pares por cada $K$.
- Por ello, se utiliza `silhouette_score(X, labels, sample_size=SAMPLE_SIZE_SILUETA, random_state=RANDOM_STATE)` con:
  - **Tamaño de muestra (`SAMPLE_SIZE_SILUETA`):** `3000` píxeles por evaluación.
  - **Semilla (`RANDOM_STATE`):** `42` (garantiza que exactamente la misma muestra pseudoaleatoria de píxeles se utilice al comparar distintos valores de $K$ y distintas razones $\lambda/\beta$).

In [ ]:
# Definición de las 10 combinaciones de (beta, lambda) con razones lambda/beta distintas
COMBINACIONES_BETA_LAMBDA = [
    {'id_config': 'C01_solo_color', 'beta': 1.0, 'lam': 0.00, 'razon_lam_beta': '0.00 (Solo color)'},
    {'id_config': 'C02_lam_0.10',   'beta': 1.0, 'lam': 0.10, 'razon_lam_beta': '0.10'},
    {'id_config': 'C03_lam_0.25',   'beta': 1.0, 'lam': 0.25, 'razon_lam_beta': '0.25'},
    {'id_config': 'C04_lam_0.50',   'beta': 1.0, 'lam': 0.50, 'razon_lam_beta': '0.50'},
    {'id_config': 'C05_lam_0.75',   'beta': 1.0, 'lam': 0.75, 'razon_lam_beta': '0.75'},
    {'id_config': 'C06_lam_1.00',   'beta': 1.0, 'lam': 1.00, 'razon_lam_beta': '1.00'},
    {'id_config': 'C07_lam_1.50',   'beta': 1.0, 'lam': 1.50, 'razon_lam_beta': '1.50'},
    {'id_config': 'C08_lam_2.00',   'beta': 1.0, 'lam': 2.00, 'razon_lam_beta': '2.00'},
    {'id_config': 'C09_lam_3.00',   'beta': 1.0, 'lam': 3.00, 'razon_lam_beta': '3.00'},
    {'id_config': 'C10_solo_pos',   'beta': 0.0, 'lam': 1.00, 'razon_lam_beta': 'inf (Solo posición)'},
]

assert len(COMBINACIONES_BETA_LAMBDA) == 10, "Deben ser exactamente 10 combinaciones de beta y lambda."
display(pd.DataFrame(COMBINACIONES_BETA_LAMBDA))

In [ ]:
def seleccionar_k_por_silueta(
    img,
    beta,
    lam,
    k_range=K_RANGE,
    sample_size=SAMPLE_SIZE_SILUETA,
    random_state=RANDOM_STATE,
    pipe_base=None,
):
    '''
    Adaptado de `evaluar_k` (P11_KMeans):
    1. Transforma la imagen usando el pipeline de representación con los pesos (beta, lam).
    2. Evalúa KMeans para cada K en `k_range` (2 a 8).
    3. Calcula el coeficiente de silueta sobre una muestra reproducible de píxeles
       (`sample_size`, `random_state`).
    4. Selecciona el K* con mayor coeficiente de silueta y retorna el mejor resultado junto
       con el historial por cada K.
    '''
    if pipe_base is None:
        pipe_base = crear_pipeline_representacion(beta=beta, lam=lam)
    else:
        pipe_base.set_params(ponderador__beta=beta, ponderador__lam=lam)

    X_rep = pipe_base.fit_transform(img)
    # También extraemos las características en [0, 1] sin ponderar para poder caracterizar
    # los grupos de forma interpretable (color RGB en [0,1] y posición x,y en [0,1])
    X_unit = pipe_base.named_steps['extractor'].transform(img)

    n_efectivo = min(sample_size, len(X_rep)) if sample_size is not None else None

    historial_k = []
    mejor_resultado = None
    mejor_silueta = -np.inf

    for k in k_range:
        kmeans = KMeans(n_clusters=k, n_init=10, max_iter=300, random_state=random_state)
        labels = kmeans.fit_predict(X_rep)
        sil = float(silhouette_score(X_rep, labels, sample_size=n_efectivo, random_state=random_state))
        inercia = float(kmeans.inertia_)

        historial_k.append({
            'K': k,
            'silueta': sil,
            'inercia': inercia,
        })

        if sil > mejor_silueta:
            mejor_silueta = sil
            mejor_resultado = {
                'k_optimo': k,
                'silueta_optima': sil,
                'inercia_optima': inercia,
                'labels': labels,
                'centroids_rep': kmeans.cluster_centers_,
                'X_unit': X_unit,
                'shape_img': img.shape[:2],
            }

    mejor_resultado['historial_k'] = pd.DataFrame(historial_k)
    return mejor_resultado


def graficar_codo_y_silueta(df_historial_k, titulo=""):
    '''
    Replicado de `evaluar_k` en P11_KMeans:
    Grafica el método del codo (inercia) y el coeficiente de silueta en función de K,
    destacando el K óptimo seleccionado por silueta.
    '''
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)

    axes[0].plot(df_historial_k['K'], df_historial_k['inercia'], marker='o', color='#4C6EF5')
    axes[0].set(xlabel='Número de clústeres (K)', ylabel='Inercia', title='Método del codo (Inercia)')
    axes[0].set_xticks(df_historial_k['K'])
    axes[0].grid(alpha=0.3)

    idx_max = df_historial_k['silueta'].idxmax()
    k_opt = int(df_historial_k.loc[idx_max, 'K'])
    sil_opt = df_historial_k.loc[idx_max, 'silueta']

    axes[1].plot(df_historial_k['K'], df_historial_k['silueta'], marker='o', color='#F76707')
    axes[1].scatter([k_opt], [sil_opt], color='#E03131', s=90, zorder=5, label=f'K* = {k_opt} (Silueta = {sil_opt:.3f})')
    axes[1].set(xlabel='Número de clústeres (K)', ylabel='Coeficiente de Silueta', title='Selección de K por Silueta')
    axes[1].set_xticks(df_historial_k['K'])
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    if titulo:
        plt.suptitle(titulo, fontsize=12, fontweight='bold')
    plt.show()


def grafico_silueta_muestra(X_rep, etiquetas, titulo="Perfil de silueta por clúster", sample_size=SAMPLE_SIZE_SILUETA, random_state=RANDOM_STATE):
    '''
    Adaptado de `grafico_silueta()` en P12_DBSCAN_Aglomerativo:
    Grafica el perfil de silueta de cada píxel (sobre la muestra reproducible) agrupado por clúster.
    '''
    rng = np.random.default_rng(random_state)
    if sample_size is not None and len(X_rep) > sample_size:
        idx = rng.choice(len(X_rep), size=sample_size, replace=False)
        X_c, etq = X_rep[idx], etiquetas[idx]
    else:
        X_c, etq = X_rep, etiquetas

    valores_silueta = silhouette_samples(X_c, etq)
    promedio = valores_silueta.mean()
    clusters = np.unique(etq)

    plt.figure(figsize=(9, 4.2))
    y_inferior = 10
    for i, c in enumerate(clusters):
        valores_c = np.sort(valores_silueta[etq == c])
        y_superior = y_inferior + len(valores_c)
        color = cm.nipy_spectral(float(i) / len(clusters))
        plt.fill_betweenx(np.arange(y_inferior, y_superior), 0, valores_c, facecolor=color, edgecolor=color, alpha=0.75)
        plt.text(-0.05, y_inferior + 0.5 * len(valores_c), f"C{c}", fontsize=9)
        y_inferior = y_superior + 10

    plt.axvline(promedio, color='red', linestyle='--', label=f'Silueta promedio = {promedio:.3f}')
    plt.xlabel('Coeficiente de la silueta')
    plt.ylabel('Clúster')
    plt.yticks([])
    plt.title(titulo)
    plt.legend(loc='best')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## <a name="sec4"></a> 4. Actividad 3 — Construcción de la máscara de la lesión (15%)

> **Enunciado:** *K-medias agrupa los píxeles, pero no identifica cuáles corresponden a la lesión. Diseñen y justifiquen un criterio automático que, a partir de las características de los grupos (por ejemplo, color, tamaño o ubicación), permita identificar los que representan la lesión y construir una máscara binaria. Pueden comparar dos criterios.*

### Diseño y justificación de los dos criterios automáticos propuestos
En imágenes dermatoscópicas, los grupos obtenidos por K-medias pueden corresponder a:
1. **Lesión pigmentada:** suele presentar **menor luminancia** (colores más oscuros, menor promedio de $R, G, B$) y ubicarse **cerca del centro** de la imagen $(x \approx 0.5, y \approx 0.5)$, ya que el operador centra el dermatoscopio sobre la lesión.
2. **Piel circundante (fondo):** suele tener mayor luminancia (tonos claros/rosados) y rodear la lesión, ocupando la mayor parte del **borde exterior (marco periférico)** de la imagen.
3. **Artefactos de borde / viñeteo circular del dermatoscopio:** en varias imágenes dermatoscópicas existen esquinas completamente negras producidas por el lente del instrumento. Aunque son muy oscuras, se distinguen fácilmente de la lesión porque se concentran en los **bordes y esquinas** de la imagen (alta distancia al centro y alta proporción de píxeles en el marco exterior).

Con base en estas propiedades físicas del dominio, implementamos y comparamos **dos criterios automáticos** (que **nunca** consultan la máscara de referencia):

- **Criterio A (`criterio_1_unicapa` — Selección del clúster central más oscuro con filtro de viñeteo):**
  1. Para cada clúster $c \in \{0, \dots, K-1\}$, se calcula a partir de sus píxeles en $[0, 1]$:
     - Su **oscuridad media**: $\text{Osc}(c) = 1 - \frac{\bar{R}_c + \bar{G}_c + \bar{B}_c}{3} \in [0, 1]$.
     - La **distancia media de sus píxeles al centro** $(0.5, 0.5)$ normalizada por la semidiagonal máxima $(\sqrt{0.5^2 + 0.5^2} = \frac{\sqrt{2}}{2})$: $\text{DistCentro}(c) \in [0, 1]$, de modo que su **centralidad** es $\text{Cent}(c) = 1 - \text{DistCentro}(c)$.
     - Su **fracción de presencia en el borde exterior** (marco de 5% del alto/ancho): $\text{Borde}(c)$.
  2. Se descartan como candidatos a lesión aquellos clústeres periféricos de fondo o viñeteo cuya presencia en el borde exterior sea alta ($\text{Borde}(c) > 0.25$) o cuya centralidad sea muy baja (salvo que $K=2$, donde se elige entre los dos únicos grupos).
  3. Entre los clústeres candidatos, se selecciona aquel que maximiza el puntaje combinado de oscuridad y centralidad:
     $$\text{Score}(c) = 0.65 \cdot \text{Osc}(c) + 0.35 \cdot \text{Cent}(c) - 0.50 \cdot \text{Borde}(c)$$

- **Criterio B (`criterio_2_multigrupo` — Agrupación de clústeres interiores más oscuros que la piel periférica):**
  1. Cuando $K > 2$ (especialmente al incluir peso espacial $\lambda > 0$ o en lesiones heterogéneas), una misma lesión puede quedar dividida en **más de un clúster** (por ejemplo, el centro oscuro de la lesión y su borde pigmentado, o dos mitades espaciales de la lesión).
  2. Este criterio identifica primero el color promedio de la **piel de referencia en el marco periférico** (descartando píxeles negros extremos de viñeteo con luminancia $< 0.10$).
  3. Luego selecciona **todos los clústeres interiores** (baja fracción en el borde y alta centralidad) que sean significativamente más oscuros que la piel periférica y superen un umbral relativo del puntaje máximo $\text{Score}(c)$, permitiendo reconstruir lesiones fragmentadas en varios clústeres.

In [ ]:
def caracterizar_clusters(X_unit, labels, shape_img):
    '''
    Calcula las estadísticas descriptivas de cada clúster k en el espacio normalizado [0, 1]:
    - tamaño y proporción de píxeles
    - color medio (R, G, B), luminancia media y oscuridad (1 - luminancia)
    - centroide espacial (x_med, y_med), distancia media al centro (0.5, 0.5) y centralidad
    - fracción de píxeles del clúster que caen en el marco exterior (borde del 5% de la imagen)
    - fracción del marco exterior ocupada por el clúster
    '''
    H, W = shape_img

    # Máscara del marco periférico (borde de 5% del alto y ancho, mínimo 3 píxeles)
    b_h = max(3, int(round(H * 0.05)))
    b_w = max(3, int(round(W * 0.05)))
    marco_borde = np.zeros((H, W), dtype=bool)
    marco_borde[:b_h, :] = True
    marco_borde[-b_h:, :] = True
    marco_borde[:, :b_w] = True
    marco_borde[:, -b_w:] = True
    marco_flat = marco_borde.ravel()
    total_borde = marco_flat.sum()

    # Distancia de cada píxel al centro (0.5, 0.5) normalizada a [0, 1] (dividiendo por sqrt(0.5^2 + 0.5^2))
    dist_centro_pixel = np.sqrt((X_unit[:, 3] - 0.5) ** 2 + (X_unit[:, 4] - 0.5) ** 2) / np.sqrt(0.5)

    filas = []
    for c in np.unique(labels):
        m = (labels == c)
        rgb_c = X_unit[m, :3]
        xy_c = X_unit[m, 3:5]
        lum_media = float(rgb_c.mean())
        oscuridad = 1.0 - lum_media
        dist_media = float(dist_centro_pixel[m].mean())
        centralidad = 1.0 - dist_media

        en_borde_propio = float(marco_flat[m].mean())              # Qué fracción de este clúster está en el borde
        en_borde_total = float(marco_flat[m].sum() / total_borde)  # Qué fracción del borde total ocupa este clúster

        score_lesion = 0.65 * oscuridad + 0.35 * centralidad - 0.50 * en_borde_propio

        filas.append({
            'cluster': int(c),
            'n_pixeles': int(m.sum()),
            'prop_imagen': float(m.mean()),
            'R_med': float(rgb_c[:, 0].mean()),
            'G_med': float(rgb_c[:, 1].mean()),
            'B_med': float(rgb_c[:, 2].mean()),
            'luminancia': lum_media,
            'oscuridad': oscuridad,
            'x_med': float(xy_c[:, 0].mean()),
            'y_med': float(xy_c[:, 1].mean()),
            'dist_centro': dist_media,
            'centralidad': centralidad,
            'frac_en_borde': en_borde_propio,
            'frac_del_borde_total': en_borde_total,
            'score_lesion': score_lesion,
        })

    return pd.DataFrame(filas)


def construir_mascara_lesion(X_unit, labels, shape_img, criterio='criterio_1_unicapa'):
    '''
    Construye automáticamente una máscara binaria (H, W) en {0, 1} identificando el/los
    clúster(es) que corresponden a la lesión, sin utilizar la máscara de referencia.

    Parámetros:
    -----------
    criterio : str
        - 'criterio_1_unicapa': Selecciona el clúster individual con mayor puntaje de
          oscuridad y centralidad (descartando artefactos de borde/viñeteo).
        - 'criterio_2_multigrupo': Permite unir múltiples clústeres interiores oscuros
          cuando K > 2 divide la lesión en subregiones cromáticas o espaciales.
    '''
    H, W = shape_img
    df_c = caracterizar_clusters(X_unit, labels, shape_img)
    k_total = len(df_c)

    if k_total == 2:
        # Con K=2 solo hay dos grupos: se elige el de mayor score_lesion (oscuridad + centralidad - borde)
        c_elegido = int(df_c.loc[df_c['score_lesion'].idxmax(), 'cluster'])
        clusters_lesion = [c_elegido]
    else:
        # Filtramos clústeres claramente periféricos (piel de borde o viñeteo negro en las esquinas)
        candidatos = df_c[
            (df_c['frac_en_borde'] < 0.25) &
            (df_c['centralidad'] > 0.35) &
            (df_c['prop_imagen'] < 0.70)
        ].copy()

        if candidatos.empty:
            candidatos = df_c[df_c['frac_en_borde'] < df_c['frac_en_borde'].median()].copy()
        if candidatos.empty:
            candidatos = df_c.copy()

        if criterio == 'criterio_1_unicapa':
            c_mejor = int(candidatos.loc[candidatos['score_lesion'].idxmax(), 'cluster'])
            clusters_lesion = [c_mejor]

        elif criterio == 'criterio_2_multigrupo':
            # Identificamos la luminancia de referencia del fondo (clústeres con mayor presencia en el borde,
            # excluyendo viñeteo negro extremo con luminancia < 0.10)
            fondo_borde = df_c[(df_c['frac_del_borde_total'] > 0.15) & (df_c['luminancia'] >= 0.10)]
            if fondo_borde.empty:
                lum_fondo = float(df_c['luminancia'].max())
            else:
                lum_fondo = float((fondo_borde['luminancia'] * fondo_borde['frac_del_borde_total']).sum()
                                  / fondo_borde['frac_del_borde_total'].sum())

            max_score = float(candidatos['score_lesion'].max())
            seleccionados = candidatos[
                (candidatos['luminancia'] < lum_fondo - 0.04) &
                (candidatos['score_lesion'] >= max_score - 0.12) &
                (candidatos['frac_en_borde'] < 0.15)
            ]

            # Si la unión supera el 65% de la imagen o queda vacía, nos quedamos con el mejor candidato
            if seleccionados.empty or seleccionados['prop_imagen'].sum() > 0.65:
                clusters_lesion = [int(candidatos.loc[candidatos['score_lesion'].idxmax(), 'cluster'])]
            else:
                clusters_lesion = seleccionados['cluster'].astype(int).tolist()
        else:
            raise ValueError(f"Criterio desconocido: {criterio}")

    mask_estimada = np.isin(labels.reshape(H, W), clusters_lesion).astype(np.uint8)
    return mask_estimada, clusters_lesion, df_c

In [ ]:
def visualizar_grupos_y_mascaras(img, mask_ref, res_kmeans, titulo=""):
    '''
    Adaptado de `get_img_masks` y `spatial_img_segmentation` (P13_Segmentación de imágenes):
    Muestra la imagen original, la imagen reconstruida con los colores promedio de cada clúster,
    el mapa de clústeres y las máscaras binarias generadas por los dos criterios frente a la de referencia.
    '''
    H, W = res_kmeans['shape_img']
    labels = res_kmeans['labels']
    X_unit = res_kmeans['X_unit']
    k_opt = res_kmeans['k_optimo']

    mask_c1, cl_c1, df_c = construir_mascara_lesion(X_unit, labels, (H, W), criterio='criterio_1_unicapa')
    mask_c2, cl_c2, _ = construir_mascara_lesion(X_unit, labels, (H, W), criterio='criterio_2_multigrupo')

    # Reconstrucción cromática por clúster (color promedio RGB de cada grupo)
    img_reconstruida = np.zeros_like(img)
    for _, row in df_c.iterrows():
        c = int(row['cluster'])
        color_rgb = np.round([row['R_med'] * 255, row['G_med'] * 255, row['B_med'] * 255]).astype(np.uint8)
        img_reconstruida[labels.reshape(H, W) == c] = color_rgb

    dice_c1 = f1_score(mask_ref.ravel(), mask_c1.ravel(), pos_label=1, zero_division=0)
    dice_c2 = f1_score(mask_ref.ravel(), mask_c2.ravel(), pos_label=1, zero_division=0)

    fig, axes = plt.subplots(1, 6, figsize=(18, 3.3), constrained_layout=True)
    axes[0].imshow(img)
    axes[0].set_title("Imagen original")
    axes[1].imshow(img_reconstruida)
    axes[1].set_title(f"Color por clúster (K*={k_opt})")
    axes[2].imshow(labels.reshape(H, W), cmap='tab10')
    axes[2].set_title(f"Mapa de grupos (K*={k_opt})")
    axes[3].imshow(mask_c1, cmap='gray', vmin=0, vmax=1)
    axes[3].set_title(f"Criterio 1 (cl={cl_c1})\nDice = {dice_c1:.3f}")
    axes[4].imshow(mask_c2, cmap='gray', vmin=0, vmax=1)
    axes[4].set_title(f"Criterio 2 (cl={cl_c2})\nDice = {dice_c2:.3f}")
    axes[5].imshow(mask_ref, cmap='gray', vmin=0, vmax=1)
    axes[5].set_title("Máscara de referencia")

    for ax in axes:
        ax.axis('off')

    if titulo:
        plt.suptitle(titulo, fontsize=12, fontweight='bold')
    plt.show()
    return df_c

## <a name="sec5"></a> 5. Actividad de verificación obligatoria con IAG (10%)

> **Enunciado:** *Antes de elegir la configuración final (actividad 4), pidan a una herramienta de IAG que les recomiende valores de $\beta$, $\lambda$ y $K$, y un criterio para identificar los grupos que corresponden a la lesión y construir la máscara en imágenes dermatoscópicas. Su consulta **no debe anexar imágenes o mencionar el nombre del conjunto (ISIC)**, debe describir las imágenes. Adjunten el historial de la conversación con la IAG donde se muestre el prompt, la respuesta y sucesivas interacciones. Pongan a prueba la recomendación con sus imágenes de ajuste. En la sección "Uso de herramientas de IA generativa" respondan: ¿la recomendación funcionó en sus imágenes? ¿En qué acertó y en qué falló? ¿Por qué la IAG no podía anticipar el comportamiento del método en sus imágenes?*

### 5.1 Historial de la consulta a la herramienta de IAG

**Prompt enviado a la IAG (sin mencionar ISIC ni adjuntar imágenes):**
> *"Estamos desarrollando un método no supervisado de segmentación de lesiones cutáneas sobre fotografías tomadas con dermatoscopio (donde suele observarse una lesión pigmentada rodeada de piel, a veces con vello, sombras en las esquinas por el lente o iluminación desigual). Representamos cada píxel con el vector $[\beta \cdot R, \beta \cdot G, \beta \cdot B, \lambda \cdot x, \lambda \cdot y]$, donde $R, G, B, x, y$ están normalizados en $[0, 1]$, y aplicamos K-medias. Sin ver las imágenes, ¿qué valores concretos de $\beta$, $\lambda$ y $K$ nos recomiendas usar, y qué regla o criterio automático propones para decidir cuál(es) de los $K$ clústeres corresponden a la lesión y construir la máscara binaria?"*

**Respuesta de la IAG:**
> *(Pegar aquí o complementar la respuesta textual de la IAG e interacciones adicionales si las hubo. A continuación se deja parametrizada una recomendación típica de IAG: $\beta = 1.0$, $\lambda = 0.5$, $K = 3$ fijo, y elegir como lesión el clúster más oscuro —menor promedio RGB— o el más oscuro que no toque las esquinas).*

### 5.2 Prueba experimental de la recomendación de la IAG sobre las 15 imágenes de ajuste

In [ ]:
# Parámetros recomendados por la IAG para poner a prueba en las 15 imágenes de ajuste
BETA_IAG = 1.0
LAM_IAG = 0.5
K_IAG = 3  # K fijo recomendado por la IAG


def mascara_criterio_iag_mas_oscuro(X_unit, labels, shape_img):
    '''
    Criterio simple que suele recomendar la IAG en primera instancia:
    seleccionar como lesión el clúster con menor luminancia promedio (el más oscuro en RGB).
    '''
    H, W = shape_img
    df_c = caracterizar_clusters(X_unit, labels, shape_img)
    c_mas_oscuro = int(df_c.loc[df_c['luminancia'].idxmin(), 'cluster'])
    return (labels.reshape(H, W) == c_mas_oscuro).astype(np.uint8), [c_mas_oscuro]


def evaluar_recomendacion_iag(datos_ajuste_dict, beta=BETA_IAG, lam=LAM_IAG, k_fijo=K_IAG):
    '''Evalúa la configuración recomendada por la IAG sobre las 15 imágenes de ajuste.'''
    pipe = crear_pipeline_completo(beta=beta, lam=lam, n_clusters=k_fijo, random_state=RANDOM_STATE)
    filas = []
    for isic_id, item in datos_ajuste_dict.items():
        img, mask_ref = item['img'], item['mask']
        H, W = item['shape']
        pipe.fit(img)
        labels = pipe.named_steps['kmeans'].labels_
        X_rep = pipe[:-1].transform(img)
        X_unit = pipe.named_steps['extractor'].transform(img)
        sil = float(silhouette_score(X_rep, labels, sample_size=min(SAMPLE_SIZE_SILUETA, len(X_rep)), random_state=RANDOM_STATE))

        # 1) Criterio ingenuo de la IAG (clúster más oscuro sin filtro de esquinas)
        mask_iag_osc, _ = mascara_criterio_iag_mas_oscuro(X_unit, labels, (H, W))
        # 2) Nuestro Criterio 1 (oscuridad + centralidad con filtro de borde) usando el K_IAG
        mask_iag_c1, _, _ = construir_mascara_lesion(X_unit, labels, (H, W), criterio='criterio_1_unicapa')

        filas.append({
            'Imagen': isic_id,
            'K (IAG)': k_fijo,
            'Silueta': round(sil, 4),
            'Dice (IAG: clúster más oscuro)': round(f1_score(mask_ref.ravel(), mask_iag_osc.ravel(), pos_label=1, zero_division=0), 4),
            'Dice (IAG params + Criterio 1 nuestro)': round(f1_score(mask_ref.ravel(), mask_iag_c1.ravel(), pos_label=1, zero_division=0), 4),
            'V-measure': round(v_measure_score(mask_ref.ravel(), labels), 4),
        })
    df_res = pd.DataFrame(filas)
    return df_res


if DATOS_CARGADOS:
    df_iag = evaluar_recomendacion_iag(datos_ajuste)
    display(df_iag)
    print("Promedios sobre las 15 imágenes de ajuste con la recomendación de la IAG:")
    display(df_iag.mean(numeric_only=True).round(4).to_frame('Promedio Ajuste (IAG)').T)

## <a name="sec6"></a> 6. Actividad 4 — Ajuste del método sobre las 15 imágenes de ajuste (20%)

> **Enunciado:** *Utilizando únicamente las 15 imágenes de ajuste, evalúen las combinaciones de $\beta$ y $\lambda$ y los criterios de construcción de la máscara propuestos. Para cada configuración, calculen el coeficiente de Dice entre la máscara estimada y la de referencia, y el V-measure entre los grupos obtenidos y las etiquetas de referencia. Utilicen el Dice promedio como criterio principal para seleccionar la configuración final y el V-measure como información complementaria. Revisen también los casos de fallo y justifiquen su elección. Visualicen la imagen original y los grupos obtenidos para las dos configuraciones con mayor Dice promedio, así como para las configuraciones que utilizan únicamente color o posición. La configuración final debe incluir los valores de $\beta$ y $\lambda$ y el criterio de construcción de la máscara.*

### Métricas de evaluación
1. **Coeficiente de Dice (criterio principal de selección):** mide la superposición espacial a nivel de píxel entre la máscara de referencia $X$ y la máscara estimada $Y$:
   $$\text{DICE} = \frac{2|X \cap Y|}{|X| + |Y|}$$
   Equivale al `f1_score` binario considerando la lesión (`1`) como clase positiva.
2. **V-measure (información complementaria):** media armónica entre homogeneidad y completitud (`v_measure_score(mask_ref.ravel(), labels)`), que evalúa qué tan puros son los $K$ grupos respecto a las etiquetas de referencia sin depender del criterio posterior de binarización.

In [ ]:
def calcular_metricas_segmentacion(mask_ref, mask_est, labels_cluster):
    '''
    Calcula las métricas de evaluación entre la máscara de referencia {0, 1},
    la máscara binaria estimada {0, 1} y las etiquetas de agrupación de K-medias.
    '''
    y_true = mask_ref.ravel()
    y_pred = mask_est.ravel()
    return {
        'Dice': float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'V-measure': float(v_measure_score(y_true, labels_cluster)),
        'Precision': float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        'Recall': float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
    }


def ejecutar_experimento_ajuste(
    datos_ajuste_dict,
    combinaciones=COMBINACIONES_BETA_LAMBDA,
    criterios=('criterio_1_unicapa', 'criterio_2_multigrupo'),
    k_range=K_RANGE,
    sample_size=SAMPLE_SIZE_SILUETA,
    random_state=RANDOM_STATE,
):
    '''
    Ejecuta el protocolo completo de ajuste sobre las 15 imágenes de ajuste:
    - Para cada imagen (15) y cada combinación de (beta, lambda) (10):
      selecciona K* en {2..8} maximizando el coeficiente de silueta.
    - Para cada criterio de construcción de máscara (2):
      construye la máscara binaria y calcula Dice, V-measure, Precisión y Recall.
    Retorna:
      - df_detalle: resultados por (imagen, combinación, criterio)
      - df_historial_k_todo: silueta e inercia para todos los K explorados
      - cache_agrupaciones: diccionario {(isic_id, id_config): res_kmeans} para visualización rápida
    '''
    t0 = time.perf_counter()
    filas_detalle = []
    filas_k = []
    cache_agrupaciones = {}
    pipe_rep = crear_pipeline_representacion()

    total_pasos = len(datos_ajuste_dict) * len(combinaciones)
    paso = 0

    for isic_id, item in datos_ajuste_dict.items():
        img = item['img']
        mask_ref = item['mask']
        H, W = item['shape']

        for comb in combinaciones:
            paso += 1
            id_cfg = comb['id_config']
            beta, lam = comb['beta'], comb['lam']

            res_km = seleccionar_k_por_silueta(
                img,
                beta=beta,
                lam=lam,
                k_range=k_range,
                sample_size=sample_size,
                random_state=random_state,
                pipe_base=pipe_rep,
            )
            cache_agrupaciones[(isic_id, id_cfg)] = res_km

            # Guardar historial de todos los K (2..8) para responder la Pregunta 2 del análisis
            for _, row_k in res_km['historial_k'].iterrows():
                filas_k.append({
                    'Imagen': isic_id,
                    'id_config': id_cfg,
                    'beta': beta,
                    'lam': lam,
                    'razon_lam_beta': comb['razon_lam_beta'],
                    'K': int(row_k['K']),
                    'silueta': row_k['silueta'],
                    'inercia': row_k['inercia'],
                })

            # Evaluar cada criterio de construcción de la máscara sobre el K* elegido por silueta
            for crit in criterios:
                mask_est, cl_sel, _ = construir_mascara_lesion(
                    res_km['X_unit'], res_km['labels'], (H, W), criterio=crit
                )
                met = calcular_metricas_segmentacion(mask_ref, mask_est, res_km['labels'])
                filas_detalle.append({
                    'Imagen': isic_id,
                    'id_config': id_cfg,
                    'beta': beta,
                    'lam': lam,
                    'razon_lam_beta': comb['razon_lam_beta'],
                    'criterio': crit,
                    'K_seleccionado': res_km['k_optimo'],
                    'Silueta': res_km['silueta_optima'],
                    'Dice': met['Dice'],
                    'V-measure': met['V-measure'],
                    'Precision': met['Precision'],
                    'Recall': met['Recall'],
                    'clusters_lesion': str(cl_sel),
                })

        print(f"[{paso:>3}/{total_pasos}] Procesada imagen de ajuste {isic_id} ({time.perf_counter() - t0:.1f}s)", flush=True)

    df_detalle = pd.DataFrame(filas_detalle)
    df_historial_k_todo = pd.DataFrame(filas_k)
    return df_detalle, df_historial_k_todo, cache_agrupaciones

In [ ]:
if DATOS_CARGADOS:
    df_ajuste_detalle, df_ajuste_k_todo, cache_ajuste = ejecutar_experimento_ajuste(datos_ajuste)

    # Tabla resumen de las 20 configuraciones (10 combinaciones beta/lambda x 2 criterios),
    # ordenada por Dice promedio descendente (criterio principal de selección)
    resumen_configuraciones = (
        df_ajuste_detalle
        .groupby(['id_config', 'beta', 'lam', 'razon_lam_beta', 'criterio'], as_index=False)
        .agg(
            Dice_promedio=('Dice', 'mean'),
            Dice_std=('Dice', 'std'),
            Dice_mediana=('Dice', 'median'),
            Dice_min=('Dice', 'min'),
            Dice_max=('Dice', 'max'),
            V_measure_promedio=('V-measure', 'mean'),
            Silueta_promedio=('Silueta', 'mean'),
            Precision_promedio=('Precision', 'mean'),
            Recall_promedio=('Recall', 'mean'),
            K_promedio=('K_seleccionado', 'mean'),
        )
        .sort_values('Dice_promedio', ascending=False)
        .reset_index(drop=True)
    )
    resumen_configuraciones.index = resumen_configuraciones.index + 1
    resumen_configuraciones.index.name = 'Ranking'
    display(Markdown("### Tabla comparativa de configuraciones en las 15 imágenes de ajuste (ordenada por Dice promedio)"))
    display(resumen_configuraciones.round(4))

### 6.1 Comparación gráfica de las configuraciones y selección de la configuración final

In [ ]:
def graficar_resumen_ajuste(df_resumen):
    '''Grafica el Dice promedio y el V-measure promedio en función de la razón lambda/beta y del criterio.'''
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), constrained_layout=True)

    sns.barplot(
        data=df_resumen, x='razon_lam_beta', y='Dice_promedio', hue='criterio',
        palette=['#4C6EF5', '#FF6B6B'], ax=axes[0]
    )
    axes[0].set(title='Coeficiente de Dice promedio por configuración (15 imágenes de ajuste)',
                xlabel='Razón λ / β', ylabel='Dice promedio', ylim=(0, 1))
    axes[0].tick_params(axis='x', rotation=30)

    sns.barplot(
        data=df_resumen, x='razon_lam_beta', y='V_measure_promedio', hue='criterio',
        palette=['#4C6EF5', '#FF6B6B'], ax=axes[1]
    )
    axes[1].set(title='V-measure promedio por configuración (información complementaria)',
                xlabel='Razón λ / β', ylabel='V-measure promedio', ylim=(0, 1))
    axes[1].tick_params(axis='x', rotation=30)

    plt.show()


if DATOS_CARGADOS:
    graficar_resumen_ajuste(resumen_configuraciones)

    # Selección automática de la mejor configuración según el Dice promedio en ajuste
    mejor_fila = resumen_configuraciones.iloc[0]
    segunda_fila = resumen_configuraciones.iloc[1]

    ID_CONFIG_FINAL = mejor_fila['id_config']
    BETA_FINAL = float(mejor_fila['beta'])
    LAM_FINAL = float(mejor_fila['lam'])
    CRITERIO_FINAL = mejor_fila['criterio']

    print("=" * 80)
    print("CONFIGURACIÓN FINAL SELECCIONADA (basada únicamente en las 15 imágenes de ajuste):")
    print(f" - Configuración: {ID_CONFIG_FINAL} (beta = {BETA_FINAL}, lambda = {LAM_FINAL}, razón λ/β = {mejor_fila['razon_lam_beta']})")
    print(f" - Criterio de construcción de máscara: {CRITERIO_FINAL}")
    print(f" - Dice promedio en ajuste:      {mejor_fila['Dice_promedio']:.4f} (± {mejor_fila['Dice_std']:.4f})")
    print(f" - V-measure promedio en ajuste: {mejor_fila['V_measure_promedio']:.4f}")
    print(f" - Silueta promedio en ajuste:   {mejor_fila['Silueta_promedio']:.4f}")
    print("=" * 80)

### 6.2 Visualización requerida de las configuraciones en el conjunto de ajuste

Como exige la Actividad 4:
> *"Visualicen la imagen original y los grupos obtenidos para las dos configuraciones con mayor Dice promedio, así como para las configuraciones que utilizan únicamente color o posición. Revisen también los casos de fallo y justifiquen su elección."*

In [ ]:
def visualizar_comparacion_4_configuraciones(
    isic_id,
    datos_dict,
    cache_agrupaciones,
    top1_cfg,
    top2_cfg,
):
    '''
    Para una imagen de ajuste dada, visualiza en una sola figura:
    - La imagen original y la máscara de referencia
    - Los grupos obtenidos (color por centroide y mapa de clústeres) y la máscara estimada para:
      1) Mejor configuración (Top 1 Dice promedio)
      2) Segunda mejor configuración (Top 2 Dice promedio)
      3) Únicamente color (lambda = 0, beta = 1)
      4) Únicamente posición (beta = 0, lambda = 1)
    '''
    img = datos_dict[isic_id]['img']
    mask_ref = datos_dict[isic_id]['mask']
    H, W = datos_dict[isic_id]['shape']

    configs_mostrar = [
        (f"Top 1: {top1_cfg['id_config']} (β={top1_cfg['beta']}, λ={top1_cfg['lam']}, {top1_cfg['criterio']})",
         top1_cfg['id_config'], top1_cfg['criterio']),
        (f"Top 2: {top2_cfg['id_config']} (β={top2_cfg['beta']}, λ={top2_cfg['lam']}, {top2_cfg['criterio']})",
         top2_cfg['id_config'], top2_cfg['criterio']),
        (f"Solo Color: C01_solo_color (β=1, λ=0, {top1_cfg['criterio']})",
         'C01_solo_color', top1_cfg['criterio']),
        (f"Solo Posición: C10_solo_pos (β=0, λ=1, {top1_cfg['criterio']})",
         'C10_solo_pos', top1_cfg['criterio']),
    ]

    fig, axes = plt.subplots(len(configs_mostrar), 5, figsize=(16, 3.0 * len(configs_mostrar)), constrained_layout=True)

    for i, (etiqueta, id_cfg, crit) in enumerate(configs_mostrar):
        res_km = cache_agrupaciones[(isic_id, id_cfg)]
        labels = res_km['labels']
        X_unit = res_km['X_unit']
        k_opt = res_km['k_optimo']
        sil = res_km['silueta_optima']

        mask_est, cl_sel, df_c = construir_mascara_lesion(X_unit, labels, (H, W), criterio=crit)
        met = calcular_metricas_segmentacion(mask_ref, mask_est, labels)

        # Imagen coloreada por el color promedio RGB de cada clúster
        img_col = np.zeros_like(img)
        for _, r in df_c.iterrows():
            c = int(r['cluster'])
            rgb_c = np.round([r['R_med'] * 255, r['G_med'] * 255, r['B_med'] * 255]).astype(np.uint8)
            img_col[labels.reshape(H, W) == c] = rgb_c

        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"{isic_id} — Original", fontsize=9)

        axes[i, 1].imshow(img_col)
        axes[i, 1].set_title(f"{etiqueta.split(':')[0]}\nColor grupos (K*={k_opt}, Sil={sil:.2f})", fontsize=9)

        axes[i, 2].imshow(labels.reshape(H, W), cmap='tab10')
        axes[i, 2].set_title(f"Grupos K-medias (V={met['V-measure']:.2f})", fontsize=9)

        axes[i, 3].imshow(mask_est, cmap='gray', vmin=0, vmax=1)
        axes[i, 3].set_title(f"Máscara estimada (cl={cl_sel})\nDice = {met['Dice']:.3f}", fontsize=9)

        axes[i, 4].imshow(mask_ref, cmap='gray', vmin=0, vmax=1)
        axes[i, 4].set_title("Máscara de referencia", fontsize=9)

        for j in range(5):
            axes[i, j].axis('off')

    plt.suptitle(
        f"Comparación de las 4 configuraciones requeridas en {isic_id} "
        f"(Top 1, Top 2, Solo Color y Solo Posición)",
        fontsize=12, fontweight='bold'
    )
    plt.show()


if DATOS_CARGADOS:
    # Tabla de desempeño por imagen para la configuración ganadora (para identificar mejores casos y casos de fallo)
    df_ganadora_por_img = (
        df_ajuste_detalle[
            (df_ajuste_detalle['id_config'] == ID_CONFIG_FINAL) &
            (df_ajuste_detalle['criterio'] == CRITERIO_FINAL)
        ]
        .sort_values('Dice', ascending=False)
        .reset_index(drop=True)
    )
    display(Markdown("### Desempeño por imagen de ajuste con la configuración seleccionada (de mayor a menor Dice)"))
    display(df_ganadora_por_img.round(4))

    # Visualizamos el mejor caso y el caso de fallo más pronunciado del conjunto de ajuste
    mejor_img_id = df_ganadora_por_img.iloc[0]['Imagen']
    peor_img_id = df_ganadora_por_img.iloc[-1]['Imagen']

    visualizar_comparacion_4_configuraciones(mejor_img_id, datos_ajuste, cache_ajuste, mejor_fila, segunda_fila)
    visualizar_comparacion_4_configuraciones(peor_img_id, datos_ajuste, cache_ajuste, mejor_fila, segunda_fila)

## <a name="sec7"></a> 7. Actividad 5 — Evaluación y análisis sobre las 5 imágenes de prueba (20%)

> **Enunciado:** *Apliquen la configuración seleccionada, sin modificaciones, a las cinco imágenes de prueba. Presenten una tabla con el K seleccionado, la silueta, el Dice, el V-measure, la precisión y el recall para cada imagen, considerando la lesión como clase positiva. Incluyan la imagen original, la máscara estimada y la de referencia. Comparen el Dice promedio de ajuste y prueba, e identifiquen los principales aciertos y errores. No se espera un Dice alto en todas las imágenes.*

In [ ]:
def evaluar_conjunto_prueba(
    datos_prueba_dict,
    beta_final,
    lam_final,
    criterio_final,
    k_range=K_RANGE,
    sample_size=SAMPLE_SIZE_SILUETA,
    random_state=RANDOM_STATE,
):
    '''
    Aplica la configuración final seleccionada en la Actividad 4, SIN MODIFICACIONES,
    a las 5 imágenes de prueba. Genera la tabla de métricas y las visualizaciones requeridas.
    '''
    pipe_rep = crear_pipeline_representacion(beta=beta_final, lam=lam_final)
    filas_prueba = []
    resultados_visuales = {}

    for isic_id, item in datos_prueba_dict.items():
        img = item['img']
        mask_ref = item['mask']
        H, W = item['shape']

        res_km = seleccionar_k_por_silueta(
            img,
            beta=beta_final,
            lam=lam_final,
            k_range=k_range,
            sample_size=sample_size,
            random_state=random_state,
            pipe_base=pipe_rep,
        )
        mask_est, cl_sel, df_c = construir_mascara_lesion(
            res_km['X_unit'], res_km['labels'], (H, W), criterio=criterio_final
        )
        met = calcular_metricas_segmentacion(mask_ref, mask_est, res_km['labels'])

        filas_prueba.append({
            'Imagen': isic_id,
            'K seleccionado': res_km['k_optimo'],
            'Silueta': res_km['silueta_optima'],
            'Dice': met['Dice'],
            'V-measure': met['V-measure'],
            'Precisión': met['Precision'],
            'Recall': met['Recall'],
        })
        resultados_visuales[isic_id] = {
            'img': img,
            'mask_ref': mask_ref,
            'mask_est': mask_est,
            'labels': res_km['labels'].reshape(H, W),
            'k_optimo': res_km['k_optimo'],
            'silueta': res_km['silueta_optima'],
            'metricas': met,
            'clusters_lesion': cl_sel,
            'df_clusters': df_c,
        }

    df_prueba = pd.DataFrame(filas_prueba)
    return df_prueba, resultados_visuales


def mostrar_resultados_prueba(resultados_visuales):
    '''
    Muestra para cada una de las 5 imágenes de prueba:
    1) Imagen original con contorno de referencia (verde) y estimado (rojo)
    2) Mapa de grupos K-medias
    3) Máscara estimada
    4) Máscara de referencia
    5) Mapa de aciertos y errores (Verde=VP, Rojo=FP, Azul=FN, Negro=VN)
    '''
    n = len(resultados_visuales)
    fig, axes = plt.subplots(n, 5, figsize=(16, 3.1 * n), constrained_layout=True)

    for i, (isic_id, info) in enumerate(resultados_visuales.items()):
        img = info['img']
        m_ref = info['mask_ref']
        m_est = info['mask_est']
        met = info['metricas']

        # Mapa RGB de errores: VP = Verde, FP = Rojo, FN = Azul, VN = Gris oscuro
        mapa_error = np.zeros((*m_ref.shape, 3), dtype=np.uint8)
        mapa_error[(m_ref == 1) & (m_est == 1)] = [43, 138, 62]    # VP: Verde
        mapa_error[(m_ref == 0) & (m_est == 1)] = [224, 49, 49]    # FP: Rojo (sobre-segmentación)
        mapa_error[(m_ref == 1) & (m_est == 0)] = [59, 91, 219]    # FN: Azul (sub-segmentación)
        mapa_error[(m_ref == 0) & (m_est == 0)] = [30, 30, 30]     # VN: Fondo

        axes[i, 0].imshow(img)
        axes[i, 0].contour(m_ref, levels=[0.5], colors=['#00FF66'], linewidths=1.4)
        axes[i, 0].contour(m_est, levels=[0.5], colors=['#FF3333'], linewidths=1.4, linestyles='--')
        axes[i, 0].set_title(f"{isic_id} — Original\n(Verde=Ref, Rojo=Est)", fontsize=9)

        axes[i, 1].imshow(info['labels'], cmap='tab10')
        axes[i, 1].set_title(f"Grupos (K*={info['k_optimo']}, Sil={info['silueta']:.2f})", fontsize=9)

        axes[i, 2].imshow(m_est, cmap='gray', vmin=0, vmax=1)
        axes[i, 2].set_title(f"Máscara Estimada\nDice={met['Dice']:.3f} | V={met['V-measure']:.3f}", fontsize=9)

        axes[i, 3].imshow(m_ref, cmap='gray', vmin=0, vmax=1)
        axes[i, 3].set_title("Máscara de Referencia", fontsize=9)

        axes[i, 4].imshow(mapa_error)
        axes[i, 4].set_title(f"Errores (Verde=VP, Rojo=FP, Azul=FN)\nPrec={met['Precision']:.2f} | Rec={met['Recall']:.2f}", fontsize=9)

        for j in range(5):
            axes[i, j].axis('off')

    plt.suptitle("Evaluación en las 5 imágenes de prueba (Configuración final sin modificaciones)", fontsize=13, fontweight='bold')
    plt.show()


if DATOS_CARGADOS:
    df_prueba_resultados, vis_prueba = evaluar_conjunto_prueba(
        datos_prueba,
        beta_final=BETA_FINAL,
        lam_final=LAM_FINAL,
        criterio_final=CRITERIO_FINAL,
    )
    display(Markdown("### Tabla de resultados en las 5 imágenes de prueba"))
    display(df_prueba_resultados.round(4))

    # Comparación directa de promedios entre Ajuste (15 imágenes) y Prueba (5 imágenes)
    comparacion_ajuste_prueba = pd.DataFrame({
        'Ajuste (15 imágenes)': df_ganadora_por_img[['K_seleccionado', 'Silueta', 'Dice', 'V-measure', 'Precision', 'Recall']].mean().values,
        'Prueba (5 imágenes)': df_prueba_resultados[['K seleccionado', 'Silueta', 'Dice', 'V-measure', 'Precisión', 'Recall']].mean().values,
        'Std Ajuste': df_ganadora_por_img[['K_seleccionado', 'Silueta', 'Dice', 'V-measure', 'Precision', 'Recall']].std().values,
        'Std Prueba': df_prueba_resultados[['K seleccionado', 'Silueta', 'Dice', 'V-measure', 'Precisión', 'Recall']].std().values,
    }, index=['K promedio', 'Silueta', 'Dice', 'V-measure', 'Precisión', 'Recall'])
    display(Markdown("### Comparación de desempeño promedio: Ajuste vs. Prueba"))
    display(comparacion_ajuste_prueba.round(4))

    mostrar_resultados_prueba(vis_prueba)

## <a name="sec8"></a> 8. Análisis de resultados (30%)

A continuación, se generan las tablas y gráficas auxiliares que respaldan con cifras concretas del experimento las respuestas a las cinco preguntas del enunciado.

In [ ]:
# ==============================================================================
# GRÁFICAS Y TABLAS DE SOPORTE PARA EL ANÁLISIS DE RESULTADOS (Preguntas 1, 2 y 3)
# ==============================================================================
if DATOS_CARGADOS:
    # 1. Efecto de incluir la posición por cada imagen de ajuste (Configuración final vs Solo Color vs Solo Posición)
    piv_pos = (
        df_ajuste_detalle[df_ajuste_detalle['criterio'] == CRITERIO_FINAL]
        .pivot(index='Imagen', columns='id_config', values='Dice')
    )
    df_efecto_pos = pd.DataFrame({
        'Solo Color (λ=0)': piv_pos['C01_solo_color'],
        f'Config. Final ({ID_CONFIG_FINAL})': piv_pos[ID_CONFIG_FINAL],
        'Solo Posición (β=0)': piv_pos['C10_solo_pos'],
        'Δ Dice (Final - Solo Color)': piv_pos[ID_CONFIG_FINAL] - piv_pos['C01_solo_color'],
    }).sort_values('Δ Dice (Final - Solo Color)', ascending=False)

    display(Markdown("#### Soporte Pregunta 1 — Efecto de incluir la posición de los píxeles por imagen de ajuste"))
    display(df_efecto_pos.round(4))

    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), constrained_layout=True)
    df_efecto_pos['Δ Dice (Final - Solo Color)'].plot(
        kind='barh', ax=axes[0],
        color=np.where(df_efecto_pos['Δ Dice (Final - Solo Color)'] >= 0, '#2B8A3E', '#E03131')
    )
    axes[0].axvline(0, color='black', lw=1)
    axes[0].set(title='Cambio en Dice al incluir posición (Config. Final vs Solo Color)', xlabel='Δ Dice')

    # 2. Relación entre K, Silueta, V-measure y Dice
    sub_crit = df_ajuste_detalle[df_ajuste_detalle['criterio'] == CRITERIO_FINAL]
    sc = axes[1].scatter(
        sub_crit['Silueta'], sub_crit['Dice'],
        c=sub_crit['K_seleccionado'], cmap='viridis', alpha=0.75, s=40
    )
    plt.colorbar(sc, ax=axes[1], label='K seleccionado')
    corr_sil_dice = sub_crit['Silueta'].corr(sub_crit['Dice'])
    corr_vm_dice = sub_crit['V-measure'].corr(sub_crit['Dice'])
    axes[1].set(
        title=f'Relación entre Silueta y Dice (r = {corr_sil_dice:.3f} | r(V-measure, Dice) = {corr_vm_dice:.3f})',
        xlabel='Coeficiente de Silueta', ylabel='Coeficiente de Dice'
    )
    plt.show()

### 1. ¿Qué efecto tuvo incluir la posición de los píxeles? Expliquen en qué imágenes mejoró o perjudicó la segmentación y por qué.

- **Efecto global:** *(Completar con los valores exactos de `df_efecto_pos` tras ejecutar: comparar el Dice promedio de `C01_solo_color` $(\lambda=0)$ frente a las configuraciones con peso espacial moderado $\lambda/\beta \in [0.1, 1.0]$ y frente al extremo `C10_solo_pos` $(\beta=0)$).*
- **Imágenes donde incluir la posición mejoró la segmentación ($\Delta \text{Dice} > 0$):** Cuando existen sombras periféricas, viñeteo del dermatoscopio o zonas de piel oscurecida hacia los bordes con colores similares a los de la lesión, un modelo basado únicamente en color ($\lambda = 0$) asigna esos píxeles lejanos al mismo clúster de la lesión (reduciendo la precisión). Al incorporar $\lambda \cdot x, \lambda \cdot y$ con un peso moderado, se penaliza la distancia espacial y se logra separar cromática y espacialmente la lesión central de las sombras periféricas.
- **Imágenes donde un peso alto de posición perjudicó la segmentación ($\Delta \text{Dice} < 0$):** Cuando la lesión es muy grande, alargada, irregular o cuando $\lambda / \beta$ es demasiado alto, los términos espaciales dominan la distancia euclidiana. Como K-medias induce particiones de Voronoi convexas/esféricas, un $\lambda$ elevado fuerza a que la imagen se divida en cuadrantes geométricos (como ocurre en el caso extremo $\beta = 0, \lambda = 1$), partiendo lesiones grandes en varios pedazos o agrupando piel cercana junto con la lesión.

---

### 2. ¿Qué relación encontraron entre K, la silueta y el Dice? ¿Por qué estas métricas no necesariamente conducen a las mismas decisiones?

- **Relación observada:** *(Citar el coeficiente de correlación `corr_sil_dice` y `corr_vm_dice` calculados arriba y los valores de $K$ más frecuentes elegidos por la silueta).*
- **Por qué no conducen necesariamente a las mismas decisiones:**
  1. **Naturaleza de las métricas (interna vs. externa):** El **coeficiente de silueta** es una métrica *interna* no supervisada que mide qué tan compactos y separados son los grupos en el espacio de representación $[\beta R, \beta G, \beta B, \lambda x, \lambda y]$. En cambio, el **coeficiente de Dice** es una métrica *externa* supervisada que evalúa la coincidencia espacial de la máscara binaria final con la máscara clínica del dermatólogo.
  2. **Preferencia de la silueta por $K=2$ o por el viñeteo:** En muchas imágenes dermatoscópicas, la separación más marcada en el espacio de características no ocurre entre la lesión y la piel, sino entre el **aro negro del dermatoscopio (viñeteo)** y el resto de la imagen, o entre dos mitades iluminadas. Eso produce una **silueta muy alta con $K=2$**, pero deja a la lesión mezclada con la piel sana (dando un **Dice bajo**). A la inversa, un $K=3$ o $K=4$ puede tener una silueta ligeramente menor porque las transiciones entre piel, borde de la lesión y centro de la lesión son graduales (fronteras difusas), pero permite aislar la lesión en su propio clúster y alcanzar un **Dice mucho mayor**.
  3. **Diferencia entre V-measure y Dice:** Incluso cuando los grupos tienen alta homogeneidad respecto a la lesión, el V-measure penaliza que el fondo sano se divida en varios clústeres de piel/sombra (baja completitud), mientras que el Dice solo evalúa si el subconjunto de clústeres seleccionados como lesión coincide con la región positiva.

---

### 3. ¿Cómo se comparan los resultados de ajuste y prueba? ¿Por qué es importante no modificar el método después de consultar las máscaras de prueba?

- **Comparación cuantitativa:** *(Citar los valores de la tabla `comparacion_ajuste_prueba`: Dice promedio, desviación estándar, precisión y recall en las 15 imágenes de ajuste frente a las 5 de prueba, señalando en cuáles imágenes de prueba el método acertó y en cuáles falló).*
- **Importancia del protocolo (no modificar el método tras ver prueba):**
  - El conjunto de ajuste (15 imágenes) ya se utilizó para tomar múltiples decisiones: elegir la razón $\lambda/\beta$, diseñar las reglas heurísticas del criterio de construcción de la máscara y escoger la configuración ganadora. Aunque K-medias no es un clasificador supervisado, **seleccionar hiperparámetros y reglas de decisión maximizando el Dice sobre un conjunto de imágenes con máscaras de referencia es una forma de ajuste supervisado a nivel de protocolo**.
  - Si después de observar los errores en las 5 imágenes de prueba modificáramos $\beta$, $\lambda$, el rango de $K$ o los umbrales del criterio de máscara para mejorar esas 5 imágenes, incurriríamos en **fuga de información (*data leakage*) y sobreajuste al conjunto de prueba**. La métrica resultante dejaría de ser un estimador imparcial del desempeño esperado cuando DermAlpes utilice la herramienta con nuevas fotografías en los centros de salud.

---

### 4. Identifiquen dos limitaciones de su modelo y expliquen sus posibles causas a partir de las imágenes y de la representación RGB + posición.

1. **Limitación 1 — Sensibilidad a artefactos visuales, bajo contraste y variaciones de iluminación (representación puramente local por píxel):**
   - **Causa en las imágenes y en la representación:** Cada píxel se representa únicamente por su color puntual $[R, G, B]$ y sus coordenadas $[x, y]$, sin información de textura local, gradientes de borde ni contexto morfológico. Por ello:
     - **Vello oscuro, reglas milimetradas o burbujas/marcas:** los pelos oscuros sobre la piel sana tienen valores de $[R, G, B]$ casi idénticos a la lesión pigmentada y quedan asignados al mismo grupo.
     - **Lesiones amelanóticas o de bajo contraste (tonos claros/rosados):** cuando la lesión tiene un color muy cercano al de la piel circundante, las distribuciones de color se solapan en el espacio RGB y K-medias no puede separarlas.
     - **Sombras e iluminación desigual:** el gradiente de luz del dermatoscopio altera los valores RGB de la piel sana hacia la periferia.
2. **Limitación 2 — Sesgo geométrico de K-medias hacia clústeres convexos/esféricos y dependencia de un criterio heurístico de binarización:**
   - **Causa en el algoritmo y en la representación:** K-medias minimiza la inercia euclidiana (suma de distancias cuadráticas al centroide), lo que asume implícitamente grupos convexos, isotrópicos y de varianza similar. Como se observó en la visualización 3D del espacio RGB, las nubes de píxeles en imágenes reales son **altamente anisotrópicas y elongadas**. Además, al aumentar $\lambda$, las coordenadas espaciales fuerzan particiones de Voronoi espacialmente convexas, lo cual falla en lesiones con bordes irregulares, asimétricos o con pigmentación heterogénea (donde el centro es muy oscuro y la periferia es marrón claro).

---

### 5. En el contexto de DermAlpes, ¿qué consecuencias podrían tener los falsos positivos y falsos negativos? ¿Qué sesgos y problemas de generalización deben considerarse antes de utilizar el método con nuevas poblaciones o condiciones de adquisición?

- **Consecuencias de los Falsos Positivos (FP — píxeles de piel sana o artefactos marcados como lesión, baja precisión):**
  - La máscara estimada sobreestima el área y deforma el contorno de la lesión (por ejemplo, incluyendo sombras, vello o piel circundante). Para el dermatólogo de DermAlpes que utiliza la herramienta para **medir el tamaño (diámetro/área) y evaluar la asimetría o regularidad de los bordes (criterios ABCDE)**, una sobre-segmentación puede llevar a sobreestimar el crecimiento de una lesión o percibir bordes artificialmente irregulares, generando alarmas innecesarias, remisiones prioritarias injustificadas o pérdida de confianza en la herramienta (obligando al médico a corregir manualmente la máscara).
- **Consecuencias de los Falsos Negativos (FN — píxeles de la lesión omitidos como piel sana, bajo recall):**
  - Son **clínicamente más riesgosos**: si el método recorta los bordes más claros o difusos de una lesión maligna (sub-segmentación) o falla por completo al detectar una lesión de bajo contraste, el especialista que confíe en la delimitación automática podría **subestimar el diámetro real de la lesión** o pasar por alto una zona periférica de crecimiento activo o regresión.
- **Sesgos de representación y problemas de generalización en DermAlpes:**
  1. **Sesgo de fototipo cutáneo (población de ISIC vs. pacientes rurales en Colombia):** Los repositorios públicos internacionales como ISIC 2016 están compuestos abrumadoramente por fotografías de pacientes de **piel clara (fototipos I–III de Fitzpatrick)**. En los centros de salud de municipios pequeños atendidos por DermAlpes en Colombia, existe una alta diversidad de **fototipos más oscuros (IV–VI)**. En pieles más oscuras o mestizas, el contraste de luminancia entre la piel sana y la lesión disminuye drásticamente, por lo que una heurística basada en que *"la lesión es mucho más oscura que la piel"* tendrá una tasa de fallo significativamente mayor si se despliega sin validación local.
  2. **Condiciones de adquisición y dispositivos (*domain shift*):** En los municipios pequeños, las fotografías pueden tomarse con distintos dermatoscopios acoplados a teléfonos celulares, diferente temperatura de color (balance de blancos), distinta magnificación, presencia de gel/aceite de inmersión o lesiones no centradas. Cualquier cambio en la iluminación o en el centrado afecta directamente la representación $[\beta R, \beta G, \beta B, \lambda x, \lambda y]$.

## <a name="sec9"></a> 9. Uso de herramientas de IA generativa

### 9.1 Evaluación crítica de la Actividad de Verificación con IAG
- **¿La recomendación de la IAG funcionó en las imágenes de ajuste?**
  - *(Completar con el Dice promedio obtenido en la Sección 5.2 frente al mejor Dice promedio de nuestra exploración en la Sección 6).*
- **¿En qué acertó y en qué falló?**
  - **Aciertos:** La IAG identificó razonablemente que en imágenes dermatoscópicas las lesiones pigmentadas suelen tener menor luminancia que la piel y ubicarse hacia el centro, y propuso una razón $\lambda/\beta$ moderada (por ejemplo, $\beta=1.0, \lambda=0.5$) para suavizar el ruido espacial sin destruir los bordes cromáticos.
  - **Fallos:** Al fijar un único $K$ estático para todas las imágenes o al elegir simplemente el clúster más oscuro sin filtrar el marco periférico, la recomendación de la IAG falla gravemente en imágenes que presentan **viñeteo negro en las esquinas** (donde el clúster más oscuro es el borde del lente y no la lesión, dando $\text{Dice} \approx 0$) o en imágenes donde la lesión requiere un $K$ adaptado a la complejidad cromática de la fotografía.
- **¿Por qué la IAG no podía anticipar el comportamiento del método en nuestras imágenes?**
  - Porque sin acceso a los datos reales no es posible conocer la distribución empírica de intensidades RGB, el tamaño relativo de cada lesión, la presencia de artefactos específicos (viñeteo circular, vello, burbujas, sombras laterales) ni la interacción exacta entre la geometría de Voronoi de K-medias y la morfología particular de cada una de las 15 imágenes de ajuste.

---

### 9.2 Declaración general de uso de IAG (Reglas prácticas del curso ISIS-2611)

| Herramienta | Tipo de uso |
|---|---|
| **Antigravity (Gemini)** | Estructura base del notebook a partir de las prácticas P11, P12 y P13, implementación modular de los transformadores `ExtractorPixelesImagen` y `PonderadorColorPosicion` para `Pipeline`, funciones de visualización de máscaras y soporte de depuración. |
| **Herramienta de IAG (Actividad de verificación)** | Consulta a ciegas (sin imágenes ni mención a ISIC) para obtener una recomendación inicial de $(\beta, \lambda, K)$ y criterio de máscara, evaluada experimentalmente en la Sección 5. |

### Decisiones metodológicas tomadas por el equipo

| Decisión | Alternativas consideradas | Elección y justificación del equipo |
|---|---|---|
| Normalización de $[R, G, B, x, y]$ | División por dominio teórico ($255$, $W-1$, $H-1$) vs. `MinMaxScaler` por imagen | Dominio teórico $[0, 1]$ en `ExtractorPixelesImagen` (preserva el significado físico del brillo entre imágenes claras y oscuras). |
| Diseño de las 10 combinaciones $(\beta, \lambda)$ | Parejas arbitrarias vs. barrido sistemático de la razón $\lambda/\beta$ | 10 razones $\lambda/\beta$ distintas desde $0$ (solo color) hasta $\infty$ (solo posición), aprovechando la invarianza por escala de K-medias. |
| Muestra para el coeficiente de silueta | Todos los píxeles vs. muestra aleatoria sin semilla vs. muestra reproducible fija | `SAMPLE_SIZE_SILUETA = 3000` con `RANDOM_STATE = 42` fijo para garantizar reproducibilidad exacta y eficiencia computacional. |
| Criterio de construcción de la máscara | Elegir el clúster más oscuro vs. Criterio 1 (oscuridad + centralidad con filtro de borde) vs. Criterio 2 (multigrupo por contraste periférico) | Se implementaron y compararon ambos criterios (Criterio 1 y Criterio 2) sobre las 15 imágenes de ajuste, eligiendo el de mayor Dice promedio. |

### Prompts principales utilizados
1. *"Quiero que veas el enunciado del laboratorio 4 (Laboratorio 4_Segmentacion..) y lo entiendas muy bien. Luego quiero que me crees un archivo de jupyter notebook (ipynb) con la base de la solución del laboratorio... basate en /202620/practicas para entender bien el contexto de las últimas prácticas y armar una buena base para empezar a desarrollar bien el taller."*
2. *(Prompt de la Actividad de verificación documentado textualmente en la Sección 5.1).*

### Análisis crítico y aportes propios del equipo
- **Qué partes del contenido generado fueron correctas y útiles:** La integración de la representación $[\beta R, \beta G, \beta B, \lambda x, \lambda y]$ dentro de un `Pipeline` estándar de `scikit-learn` (siguiendo P11) y las funciones de reconstrucción y visualización de máscaras (siguiendo P13).
- **Qué decisiones técnicas fueron verificadas o ajustadas por el equipo:** El equipo verificó que la interpolación de las máscaras al redimensionar fuera estrictamente por vecino más cercano (`Image.Resampling.NEAREST`), diseñó y contrastó los criterios de selección de clústeres para evitar que el viñeteo oscuro del dermatoscopio fuera confundido con la lesión, ejecutó los experimentos sobre las 15 imágenes de ajuste y redactó el análisis crítico de resultados y el video para DermAlpes.

## <a name="sec10"></a> 10. Bonos

### Bono A — Visualización del espacio de representación con PCA y t-SNE (+0.25)

> **Enunciado:** *Elijan una imagen y apliquen PCA o t-SNE a una muestra de píxeles representados mediante $[\beta \cdot R, \beta \cdot G, \beta \cdot B, \lambda \cdot x, \lambda \cdot y]$. Presenten dos visualizaciones, coloreadas por clúster y por máscara de referencia. Interpreten coincidencias, diferencias y limitaciones de la proyección.*

In [ ]:
def ejecutar_bono_a_proyeccion(
    isic_id,
    datos_dict,
    beta=1.0,
    lam=0.5,
    n_sample=2500,
    random_state=RANDOM_STATE,
):
    '''
    Aplica PCA (2D) y t-SNE (2D) a una muestra reproducible de píxeles representados mediante
    [beta*R, beta*G, beta*B, lam*x, lam*y], y grafica las dos visualizaciones coloreadas:
    1) Por clúster asignado por K-medias (con K* seleccionado por silueta)
    2) Por máscara de referencia (Lesión vs. Piel/Fondo)
    '''
    img = datos_dict[isic_id]['img']
    mask_ref = datos_dict[isic_id]['mask'].ravel()

    res_km = seleccionar_k_por_silueta(img, beta=beta, lam=lam, random_state=random_state)
    pipe_rep = crear_pipeline_representacion(beta=beta, lam=lam)
    X_rep = pipe_rep.fit_transform(img)
    labels = res_km['labels']
    k_opt = res_km['k_optimo']

    rng = np.random.default_rng(random_state)
    idx = rng.choice(len(X_rep), size=min(n_sample, len(X_rep)), replace=False)
    X_sub = X_rep[idx]
    labels_sub = labels[idx]
    mask_sub = mask_ref[idx]

    # 1. Proyección lineal con PCA (2 componentes)
    pca = PCA(n_components=2, random_state=random_state)
    Z_pca = pca.fit_transform(X_sub)
    var_exp = pca.explained_variance_ratio_ * 100

    # 2. Proyección no lineal con t-SNE (2 componentes)
    tsne = TSNE(n_components=2, perplexity=30, random_state=random_state, init='pca', learning_rate='auto')
    Z_tsne = tsne.fit_transform(X_sub)

    fig, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)

    # PCA coloreado por clúster y por máscara de referencia
    sc1 = axes[0, 0].scatter(Z_pca[:, 0], Z_pca[:, 1], c=labels_sub, cmap='tab10', s=10, alpha=0.75)
    axes[0, 0].set(
        title=f"PCA (2D) — Coloreado por Clúster K-medias (K*={k_opt})",
        xlabel=f"PC1 ({var_exp[0]:.1f}% var.)", ylabel=f"PC2 ({var_exp[1]:.1f}% var.)"
    )
    plt.colorbar(sc1, ax=axes[0, 0], label='Clúster')

    for val, etq, col in [(0, 'Fondo / Piel (0)', '#4C6EF5'), (1, 'Lesión (1)', '#E03131')]:
        m = mask_sub == val
        axes[0, 1].scatter(Z_pca[m, 0], Z_pca[m, 1], c=col, label=etq, s=10, alpha=0.65)
    axes[0, 1].set(
        title="PCA (2D) — Coloreado por Máscara de Referencia",
        xlabel=f"PC1 ({var_exp[0]:.1f}% var.)", ylabel=f"PC2 ({var_exp[1]:.1f}% var.)"
    )
    axes[0, 1].legend()

    # t-SNE coloreado por clúster y por máscara de referencia
    sc2 = axes[1, 0].scatter(Z_tsne[:, 0], Z_tsne[:, 1], c=labels_sub, cmap='tab10', s=10, alpha=0.75)
    axes[1, 0].set(
        title=f"t-SNE (2D) — Coloreado por Clúster K-medias (K*={k_opt})",
        xlabel="Dimensión t-SNE 1", ylabel="Dimensión t-SNE 2"
    )
    plt.colorbar(sc2, ax=axes[1, 0], label='Clúster')

    for val, etq, col in [(0, 'Fondo / Piel (0)', '#4C6EF5'), (1, 'Lesión (1)', '#E03131')]:
        m = mask_sub == val
        axes[1, 1].scatter(Z_tsne[m, 0], Z_tsne[m, 1], c=col, label=etq, s=10, alpha=0.65)
    axes[1, 1].set(
        title="t-SNE (2D) — Coloreado por Máscara de Referencia",
        xlabel="Dimensión t-SNE 1", ylabel="Dimensión t-SNE 2"
    )
    axes[1, 1].legend()

    plt.suptitle(
        f"Bono A — Espacio de representación [βR, βG, βB, λx, λy] en {isic_id} (β={beta}, λ={lam}, n={len(X_sub)})",
        fontsize=13, fontweight='bold'
    )
    plt.show()


if DATOS_CARGADOS:
    ejecutar_bono_a_proyeccion(
        IMAGENES_AJUSTE[0],
        datos_ajuste,
        beta=BETA_FINAL,
        lam=LAM_FINAL,
    )

**Interpretación del Bono A (PCA y t-SNE):**
- **Coincidencias y diferencias:** En la proyección PCA se observa que K-medias traza fronteras lineales (hiperplanos de Voronoi) que cortan de forma nítida la nube continua de píxeles. Al contrastar con el gráfico coloreado por la máscara de referencia, el clúster asignado a la lesión coincide con el extremo de menor luminancia y mayor centralidad, pero existe una zona de transición continua (bordes difusos de la lesión) donde los píxeles de piel y lesión se superponen.
- **Limitaciones de la proyección:** PCA es una proyección lineal ortogonal que conserva la mayor varianza global, pero al comprimir 5 dimensiones ($3$ cromáticas $+ 2$ espaciales) en 2 ejes, puede superponer clústeres que estaban separados en las dimensiones restantes. Por su parte, t-SNE preserva vecindades locales no lineales, pero distorsiona las distancias globales y los tamaños relativos de los grupos, pudiendo crear la ilusión visual de islas separadas donde en realidad hay un gradiente continuo de píxeles.

---

### Bono B — Investigación y comparación con Mean Shift (+0.25)

> **Enunciado:** *Investiguen mean shift y aplíquenlo a una imagen de ajuste, usando una resolución y un tamaño de muestra que permitan ejecutarlo. Construyan una máscara binaria y comparen con K-medias mediante Dice, visualización y tiempo de ejecución. Expliquen cómo funciona el ancho de banda y por qué mean shift no requiere fijar K. No es necesario repetir el protocolo completo en todas las imágenes.*

#### Fundamento teórico de Mean Shift
1. **¿Por qué Mean Shift no requiere fijar $K$?**
   Mean Shift es un algoritmo no paramétrico basado en **estimación de densidad por núcleos (*Kernel Density Estimation*, KDE)**. En lugar de partir de $K$ centroides predefinidos, coloca una ventana (kernel plano o gaussiano de radio $h$, llamado **ancho de banda** o `bandwidth`) sobre cada punto (o sobre una grilla de semillas) y desplaza iterativamente cada ventana hacia el **promedio local (centro de masa)** de los puntos dentro de su vecindad —es decir, asciende por el gradiente de densidad hasta converger a un **modo local (máximo de densidad)**. Todos los puntos que convergen al mismo modo local forman un clúster. Así, el número de grupos $K$ emerge automáticamente del número de modos de la distribución de píxeles.
2. **¿Cómo funciona el ancho de banda (`bandwidth`)?**
   El hiperparámetro `bandwidth` ($h$) controla la escala espacial/cromática de la ventana de búsqueda (análogamente a `eps` en DBSCAN, visto en **P12**):
   - Un **ancho de banda pequeño** ($h \downarrow$) produce una estimación de densidad muy rugosa con muchos modos locales, fragmentando la imagen en **muchos clústeres pequeños** (sobre-segmentación).
   - Un **ancho de banda grande** ($h \uparrow$) suaviza la función de densidad y fusiona modos cercanos, produciendo **pocos clústeres grandes** (sub-segmentación).
   - En `scikit-learn`, `estimate_bandwidth(X, quantile=q)` estima $h$ como el promedio de las distancias a los $q \cdot N$ vecinos más cercanos.

In [ ]:
def comparar_kmeans_vs_meanshift(
    isic_id,
    datos_dict,
    beta=1.0,
    lam=0.5,
    criterio='criterio_1_unicapa',
    quantile_bw=0.15,
    n_sample_bw=3000,
    random_state=RANDOM_STATE,
):
    '''
    Compara K-medias frente a Mean Shift sobre una imagen de ajuste en términos de:
    - Número de grupos encontrados (K* por silueta vs modos encontrados por Mean Shift)
    - Coeficiente de Dice y V-measure
    - Tiempo de ejecución (segundos)
    - Visualización lado a lado de los grupos y las máscaras binarias estimadas
    '''
    img = datos_dict[isic_id]['img']
    mask_ref = datos_dict[isic_id]['mask']
    H, W = datos_dict[isic_id]['shape']

    pipe_rep = crear_pipeline_representacion(beta=beta, lam=lam)
    X_rep = pipe_rep.fit_transform(img)
    X_unit = pipe_rep.named_steps['extractor'].transform(img)

    # 1. Ejecución de K-medias (con selección de K por silueta)
    t0_km = time.perf_counter()
    res_km = seleccionar_k_por_silueta(img, beta=beta, lam=lam, random_state=random_state, pipe_base=pipe_rep)
    mask_km, cl_km, _ = construir_mascara_lesion(X_unit, res_km['labels'], (H, W), criterio=criterio)
    t_kmeans = time.perf_counter() - t0_km
    met_km = calcular_metricas_segmentacion(mask_ref, mask_km, res_km['labels'])

    # 2. Ejecución de Mean Shift (usando bin_seeding=True para acelerar la convergencia sobre grillas de píxeles)
    t0_ms = time.perf_counter()
    bw = estimate_bandwidth(X_rep, quantile=quantile_bw, n_samples=min(n_sample_bw, len(X_rep)), random_state=random_state)
    ms = MeanShift(bandwidth=bw, bin_seeding=True)
    labels_ms = ms.fit_predict(X_rep)
    k_ms = len(np.unique(labels_ms))
    mask_ms, cl_ms, _ = construir_mascara_lesion(X_unit, labels_ms, (H, W), criterio=criterio)
    t_meanshift = time.perf_counter() - t0_ms
    sil_ms = float(silhouette_score(X_rep, labels_ms, sample_size=min(SAMPLE_SIZE_SILUETA, len(X_rep)), random_state=random_state)) if k_ms > 1 else np.nan
    met_ms = calcular_metricas_segmentacion(mask_ref, mask_ms, labels_ms)

    tabla_comp = pd.DataFrame([
        {
            'Algoritmo': 'K-medias (selección de K por silueta)',
            'Hiperparámetro clave': f"K* = {res_km['k_optimo']} (explorado 2..8)",
            'Clústeres obtenidos': res_km['k_optimo'],
            'Silueta': round(res_km['silueta_optima'], 4),
            'Dice': round(met_km['Dice'], 4),
            'V-measure': round(met_km['V-measure'], 4),
            'Precisión': round(met_km['Precision'], 4),
            'Recall': round(met_km['Recall'], 4),
            'Tiempo (s)': round(t_kmeans, 3),
        },
        {
            'Algoritmo': f'Mean Shift (quantile={quantile_bw})',
            'Hiperparámetro clave': f"bandwidth = {bw:.4f}",
            'Clústeres obtenidos': k_ms,
            'Silueta': round(sil_ms, 4),
            'Dice': round(met_ms['Dice'], 4),
            'V-measure': round(met_ms['V-measure'], 4),
            'Precisión': round(met_ms['Precision'], 4),
            'Recall': round(met_ms['Recall'], 4),
            'Tiempo (s)': round(t_meanshift, 3),
        },
    ])
    display(Markdown(f"#### Comparación cuantitativa K-medias vs. Mean Shift en `{isic_id}` (β={beta}, λ={lam})"))
    display(tabla_comp)

    fig, axes = plt.subplots(1, 6, figsize=(18, 3.3), constrained_layout=True)
    axes[0].imshow(img)
    axes[0].set_title(f"{isic_id} — Original")
    axes[1].imshow(res_km['labels'].reshape(H, W), cmap='tab10')
    axes[1].set_title(f"Grupos K-medias (K*={res_km['k_optimo']})\nTiempo: {t_kmeans:.2f}s")
    axes[2].imshow(mask_km, cmap='gray', vmin=0, vmax=1)
    axes[2].set_title(f"Máscara K-medias\nDice = {met_km['Dice']:.3f}")
    axes[3].imshow(labels_ms.reshape(H, W), cmap='tab10')
    axes[3].set_title(f"Grupos Mean Shift (K={k_ms})\nTiempo: {t_meanshift:.2f}s")
    axes[4].imshow(mask_ms, cmap='gray', vmin=0, vmax=1)
    axes[4].set_title(f"Máscara Mean Shift\nDice = {met_ms['Dice']:.3f}")
    axes[5].imshow(mask_ref, cmap='gray', vmin=0, vmax=1)
    axes[5].set_title("Máscara de Referencia")

    for ax in axes:
        ax.axis('off')
    plt.suptitle(f"Bono B — Comparación visual K-medias vs. Mean Shift en {isic_id}", fontsize=13, fontweight='bold')
    plt.show()
    return tabla_comp


if DATOS_CARGADOS:
    comparar_kmeans_vs_meanshift(
        IMAGENES_AJUSTE[0],
        datos_ajuste,
        beta=BETA_FINAL,
        lam=LAM_FINAL,
        criterio=CRITERIO_FINAL,
    )